# VTL Kernel Metrics — Canonical Gradient Field Extractor  
**Deterministic Measurement Notebook**

---
VTL Kernel Metrics functions as a coordinate system for compositional structure rather than a quality assessment tool. Individual kernel values carry no inherent aesthetic judgment—center-weighted composition (Δx ≈ 0) is not inferior to edge-weighted composition (Δx ≠ 0), and low cohesion (μ < 0.1) is not inherently worse than high cohesion (μ > 0.7). These coordinates simply locate an image in structural space.

The diagnostic power of VTL emerges from distribution analysis, not individual measurements. Compositional monoculture is revealed when semantically diverse images exhibit low variance in kernel space (e.g., Δx consistently within [-0.05, 0.05], μ within [0.01, 0.08]), indicating learned spatial priors that constrain compositional range regardless of prompt content. High kernel variance signals diverse compositional authorship; low variance signals default structural behavior. VTL measures whether compositional coordinates vary across a dataset, not whether specific coordinates are "correct."

“Failure ≠ collapse” → The Kernel Exposes Defaults, Choice, Coordinates and the Delta as Composition.

Kernel space refers to the 7-dimensional vector space formed by the ordered kernel outputs: [Δx, Δy, rᵥ, ρᵣ, μ, xₚ, θ, dₛ].

## What This Notebook Is

This notebook implements the **VTL Kernel Metrics** as a **deterministic measurement device**.  
It is **not** a model, **not** a learning system, and **not** an interpretive framework.

Given a rendered 2D image, the notebook produces a **fixed, low-dimensional kernel vector** that describes the **spatial organization of visual mass** in that image.

The output is designed to be:

- **Reproducible**
- **Model-agnostic**
- **Contrast-robust**
- **Comparable across runs, engines, and datasets**

The system operates **post-hoc** on pixel data only.  
No prompts, weights, attention maps, or semantic inference are used or implied.

---

## Core Assumption: Composition as a Gradient Field

Composition is treated as a **gradient-field problem**, not an object-recognition problem.

Given an input image: I(x,y), the following steps are applied:

1. Convert to grayscale luminance  
2. Normalize luminance to `[0, 1]`  
3. Compute first-order spatial derivatives  
4. Construct the **gradient magnitude field**

High gradient magnitude corresponds to:

- edges  
- contrast boundaries  
- texture transitions  

These regions are treated as **visual mass**, the substrate on which all kernel metrics are computed.

No learned preprocessing, denoising, or semantic filtering is applied.  
The extractor intentionally measures **raw structural behavior**.

---

## Preprocessing (Canonical and Frozen)

All preprocessing steps are **fixed** and **version-locked**:

- Grayscale luminance conversion  
- Aspect-preserving resize to canonical max dimension  
- Sobel gradient computation  
- Gradient magnitude construction  

Any change to these steps constitutes a **new device version** and breaks comparability.

---

## Adaptive Mass Mask

### Why a Mask Exists

Kernel metrics operate on **regions of structural activity**, not uniformly across the image.  
A binary **mass mask** is therefore derived from the gradient magnitude field.

The mask defines **where structure exists**.

---

### How the Mask Is Constructed

- Percentile thresholds are applied to the gradient magnitude field  
- Pixels above the high-percentile threshold are treated as **mass**  
- Lower-gradient regions define **void**

This percentile-based approach makes the system:

- Contrast-invariant (except rᵥ): All kernels use percentile normalization except void ratio, which intentionally uses an absolute threshold to measure density.
- Invariant to global contrast scaling  
- Agnostic to rendering style  

The mask is **not semantic**.  
It does **not** identify objects, figures, or meaning.

It is purely structural.

---

## Mask QA: Structural Diagnostics

The notebook includes **Mask QA utilities** to characterize the extracted mass field.

Reported diagnostics include:

- Mass fraction  
- Number of connected components  
- Largest connected component fraction  
- SHA-256 hash of the binary mask  
- Status flags: `PASS`, `WARN`, `FAIL`

  - WARN → TEXTURE_FIELD
  - PASS → REGION_FIELD
  - FAIL → INVALID

### Interpretation Boundary

Mask QA does **not** judge image quality.

Warnings such as:

- “many components”  
- “island soup”  
- “texture-driven structure”  

are **descriptive**, not corrective.

High component counts are expected in:

- highly textured images  
- dense micro-contrast fields  
- stochastic or ornamental renders  

These diagnostics surface **structural regimes**, not errors.

---

## Kernel Vector Definition

For each image, the extractor outputs a **fixed 7-dimensional kernel vector**: C = (Δx, rᵥ, ρᵣ, μ, xₚ, θ, dₛ)

Where:

- `Δx` — horizontal placement offset
- `Δy` — vertical placement offset
- `rᵥ` — void ratio (Texture Sparsity)
- `ρᵣ` — packing density  
- `μ` — cohesion  
- `xₚ` — peripheral pull  
- `θ` — orientation stability  
- `dₛ` — structural thickness  

Each metric is:

- Computed from the gradient field and its mask  
- Bounded and normalized  
- Interpretable as a structural property  

This vector is the **only sanctioned output** of the device.

---

## Determinism and Reproducibility

The notebook enforces the invariant: Same image → same mask → same kernel vector

To guarantee this:

- No randomness is used  
- All thresholds are fixed  
- All bin counts are fixed  
- Preprocessing is frozen  

Each mask emits a **SHA-256 hash** to support:

- Regression testing  
- Batch consistency checks  
- Cross-environment verification  

If two runs produce different hashes for the same image, comparability is invalidated.

---

## Explicit Non-Goals

This system does **not**:

- Infer semantic meaning  
- Detect objects or figures  
- Explain causality  
- Evaluate aesthetics  
- Prescribe composition  

It reports **observable spatial structure only**.

---

## Role in the Generative Stack

This notebook defines the **measurement substrate** that enables:
Generate → Locate → Compare → Judge → (optional) Optimize

Pipeline Overview (Canonical):
Image → Grayscale → Normalize → Sobel Gradient → Mask Construction → Kernel Extraction → 7D Vector Output

If this notebook changes, prior results are no longer comparable.

---


In [ ]:
# Cell 0 — FULL KERNEL SLATE WIPE
# Clears images, masks, reports, CSVs.
# Leaves notebook + Colab system files intact.

import os, shutil

SAFE_KEEP = {
    "sample_data",   # Colab default
}

ROOT = "/content"

for name in os.listdir(ROOT):
    if name in SAFE_KEEP:
        continue

    path = os.path.join(ROOT, name)

    try:
        if os.path.isdir(path):
            shutil.rmtree(path)
            print("🗑 removed dir:", path)
        else:
            os.remove(path)
            print("🗑 removed file:", path)
    except Exception as e:
        print("⚠️ could not remove:", path, "|", e)

print("\n✅ Kernel slate fully cleared.")
print("➡️ Upload fresh images, then run Cell 1 onward.")

Cell 1 — Setup (dependencies)

In [ ]:
# Cell 1 — Setup (Colab-stable / minimal-touch)
# Goal: avoid ABI mismatch issues WITHOUT destabilizing Colab's preinstalled stack.
# Strategy: do NOT upgrade numpy/scipy/pandas. Only install leaf deps we control.

%pip -q install --no-cache-dir \
  "scikit-image==0.24.0" \
  "opencv-python-headless==4.10.0.84"

import sys, platform
import numpy as np
import pandas as pd
import scipy
import skimage
import cv2

print("✅ Dependencies imported.")
print("Python:", sys.version.split()[0], "|", platform.platform())
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("pandas:", pd.__version__)
print("scikit-image:", skimage.__version__)
print("opencv:", cv2.__version__)
print("\n⚠️ If you changed any compiled deps, Runtime → Restart runtime, then run from Cell 2 onward.")

Cell 2 — Imports + frozen constants

In [ ]:
import os, io, zipfile, glob, hashlib
import numpy as np
import pandas as pd
import cv2

from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

from scipy.spatial import ConvexHull
from scipy.ndimage import distance_transform_edt
from skimage.morphology import skeletonize
from skimage.measure import label, regionprops, perimeter as sk_perimeter

# =========================
# FROZEN DEVICE CONSTANTS
# =========================

# Canonical working size (use one). You can change later ONLY with a version bump.
TARGET_MAX_SIDE = 1536  # keeps 1024x1536-ish landscape/portrait behavior stable

# Gradient mask percentiles (robust to global contrast scaling)
GRAD_LOW_PCT  = 85.0
GRAD_HIGH_PCT = 97.0

# Border guard (avoid edge artifacts from resizing/decoding)
EDGE_MARGIN_PX = 2

# Minimal structure threshold
MIN_MASS_FRAC  = 0.001
WARN_MASS_FRAC = 0.03

# Absolute threshold for r_v (density measurement)
R_V_ABSOLUTE_THRESHOLD = 0.15  # Test values: 0.10, 0.15, 0.20

# Orientation bins (for circular variance)
ORIENT_BINS = 8

# Determinism
EPS = 1e-9

Cell 3 — Deterministic image loading + standardization

In [ ]:
def _sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def load_image_bytes(path: str) -> bytes:
    with open(path, "rb") as f:
        return f.read()

def decode_image_bgr(image_bytes: bytes) -> np.ndarray:
    """
    Deterministic decode path:
    - uses cv2.imdecode (stable)
    - returns BGR uint8
    """
    arr = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("Failed to decode image bytes.")
    return img

def resize_max_side(img_bgr: np.ndarray, max_side: int) -> np.ndarray:
    """
    Deterministic resize: INTER_AREA for downscale, INTER_CUBIC for upscale.
    """
    h, w = img_bgr.shape[:2]
    s = max(h, w)
    if s == max_side:
        return img_bgr
    scale = max_side / float(s)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_CUBIC
    out = cv2.resize(img_bgr, (new_w, new_h), interpolation=interp)
    return out

def standardize_image_from_path(path: str, max_side: int = TARGET_MAX_SIDE) -> Tuple[np.ndarray, Dict]:
    """
    Single source of truth for image → standardized BGR.
    Returns: (img_bgr, meta)
    """
    b = load_image_bytes(path)
    img = decode_image_bgr(b)
    img = resize_max_side(img, max_side=max_side)

    meta = {
        "path": path,
        "sha256": _sha256_bytes(b),
        "h": int(img.shape[0]),
        "w": int(img.shape[1]),
        "max_side": int(max_side),
    }
    return img, meta

Cell 4 — Gradient field + canonical mass mask

In [ ]:
def bgr_to_gray_float(img_bgr: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    return gray

def sobel_gradients(gray: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    gmag = np.sqrt(gx * gx + gy * gy)
    return gx, gy, gmag

def robust_threshold_mask(gmag: np.ndarray,
                          low_pct: float = GRAD_LOW_PCT,
                          high_pct: float = GRAD_HIGH_PCT,
                          edge_margin_px: int = EDGE_MARGIN_PX) -> np.ndarray:
    """
    Canonical mass mask from gradient magnitude.
    We create a band mask: keep pixels between [P_low, P_high] and above P_low.
    (This avoids tiny high-gradient specks dominating.)
    """
    h, w = gmag.shape
    inner = gmag[edge_margin_px:h-edge_margin_px, edge_margin_px:w-edge_margin_px]
    flat = inner.reshape(-1)

    t_low  = np.percentile(flat, low_pct)
    t_high = np.percentile(flat, high_pct)

    m = (gmag >= t_low) & (gmag <= t_high)

    # enforce border margin
    if edge_margin_px > 0:
        m[:edge_margin_px, :] = False
        m[-edge_margin_px:, :] = False
        m[:, :edge_margin_px] = False
        m[:, -edge_margin_px:] = False

    return m.astype(np.uint8)  # 0/1

Cell 5 — Canonical metrics (Δx, rᵥ, ρᵣ, μ, xₚ, θ, dₛ)

In [ ]:
def metric_delta_x(mass_mask: np.ndarray) -> float:
    ys, xs = np.nonzero(mass_mask)
    if len(xs) == 0:
        return 0.0
    h, w = mass_mask.shape
    x_cent = xs.mean()
    x_frame = (w - 1) / 2.0
    return float((x_cent - x_frame) / float(w))

def metric_mass_fraction(mass_mask: np.ndarray) -> float:
    h, w = mass_mask.shape
    return float(mass_mask.sum() / float(h * w))

def metric_delta_y(mass_mask: np.ndarray) -> float:
    """
    Vertical placement offset (y-axis).
    Measures up-down balance. Indicates whether visual weight is centered,
    top-biased, or bottom-biased.

    Returns normalized offset in [-0.5, +0.5]:
    - Δy = 0 → centroid at vertical center
    - Δy < 0 → top-biased (centroid above center)
    - Δy > 0 → bottom-biased (centroid below center)
    """
    ys, xs = np.nonzero(mass_mask)
    if len(ys) == 0:
        return 0.0
    h, w = mass_mask.shape
    y_cent = ys.mean()
    y_frame = (h - 1) / 2.0
    return float((y_cent - y_frame) / float(h))

def metric_r_v(gmag: np.ndarray) -> float:
    """
    Void ratio: fraction of frame below absolute gradient threshold.

    Uses fixed threshold τ_abs = 0.15 (canonical).
    No dependence on percentile-based masking.
    """
    coverage = float((gmag >= R_V_ABSOLUTE_THRESHOLD).sum() / gmag.size)
    return float(1.0 - coverage)

def metric_rho_r(mass_mask: np.ndarray) -> float:
    """
    Packing density: mass area relative to convex hull area, scaled by 100.
    IMPORTANT: in 2D ConvexHull.volume == polygon area; ConvexHull.area == perimeter.
    """
    ys, xs = np.nonzero(mass_mask)
    if len(xs) < 3:
        return 0.0
    pts = np.stack([xs, ys], axis=1).astype(np.float64)
    hull = ConvexHull(pts)
    hull_area = float(hull.volume)  # 2D area
    mass_area = float(len(xs))
    return float(100.0 * (mass_area / (hull_area + EPS)))

def metric_mu(mass_mask: np.ndarray) -> float:
    """
    Cohesion via component size entropy (canonical per PDF spec §6.4).

    μ = p_max · (1 - H/H_max)

    Measures single-body cohesion by combining:
    - Dominance: does one component contain most mass? (p_max)
    - Fragmentation: how uniform is the size distribution? (entropy H)

    Intentionally conservative: penalizes both fragmentation (many small pieces)
    and multi-modal balance (equal-sized regions). A diptych with two equal
    regions correctly yields low μ by design.
    """
    lab = label(mass_mask > 0, connectivity=2)
    num_components = int(lab.max())

    if num_components == 0:
        return 0.0

    if num_components == 1:
        # Single component: perfect cohesion
        return 1.0

    # Component areas
    areas = []
    for i in range(1, num_components + 1):
        area = int(np.sum(lab == i))
        areas.append(area)

    areas = np.array(areas, dtype=np.float64)
    total_area = areas.sum()

    # Probabilities
    p_i = areas / (total_area + EPS)
    p_max = float(p_i.max())

    # Shannon entropy
    H = 0.0
    for p in p_i:
        if p > 0:
            H -= p * np.log2(p)

    # Maximum entropy (uniform distribution)
    H_max = np.log2(num_components)

    if H_max == 0:
        return p_max

    # Cohesion: dominance weighted by inverse entropy
    mu = p_max * (1.0 - H / H_max)

    return float(mu)

def metric_x_p(gradient_mag: np.ndarray) -> float:
    """
    Peripheral pull: edge mass ratio (canonical per PDF spec §6.5).

    xₚ = Σ_edge G(x,y) / Σ_total G(x,y)

    Measures fraction of gradient magnitude concentrated in peripheral band.
    Edges defined as outer 15% of frame per dimension (canonical frozen constant).

    High xₚ = mass concentrated at frame edges (peripheral engagement).
    Low xₚ = mass concentrated in center (radial collapse).

    Args:
        gradient_mag: Gradient magnitude field G(x,y)
    """
    h, w = gradient_mag.shape

    # Canonical peripheral band: outer 15% per dimension (PDF §6.5, Frozen Constants)
    edge_width = 0.15

    # Define edge regions
    x_edge_left = int(w * edge_width)
    x_edge_right = int(w * (1.0 - edge_width))
    y_edge_top = int(h * edge_width)
    y_edge_bottom = int(h * (1.0 - edge_width))

    # Create edge mask (True = peripheral band)
    edge_mask = np.zeros((h, w), dtype=bool)
    edge_mask[:y_edge_top, :] = True          # top band
    edge_mask[y_edge_bottom:, :] = True       # bottom band
    edge_mask[:, :x_edge_left] = True         # left band
    edge_mask[:, x_edge_right:] = True        # right band

    # Sum gradient magnitude in edges vs. total
    edge_sum = float(gradient_mag[edge_mask].sum())
    total_sum = float(gradient_mag.sum())

    if total_sum == 0:
        return 0.0

    x_p = edge_sum / total_sum

    return float(x_p)

def metric_theta(gradient_x: np.ndarray, gradient_y: np.ndarray,
                 gradient_mag: np.ndarray, mass_mask: np.ndarray,
                 num_bins: int = ORIENT_BINS) -> float:
    """
    Orientation stability via gradient orientation histogram entropy (canonical per PDF spec §6.6).

    θ = 1 - H_orient / log(N_bins)

    High θ (low entropy) = strong directional alignment in gradient field.
    Low θ (high entropy) = omnidirectional, no dominant orientation.

    Args:
        gradient_x: Sobel x-gradient (from sobel_gradients)
        gradient_y: Sobel y-gradient (from sobel_gradients)
        gradient_mag: Gradient magnitude field
        mass_mask: Binary mask defining where to measure
        num_bins: Orientation bins (CANONICAL = 8)
    """
    # Enforce canonical bin count
    if num_bins != 8:
        raise ValueError(f"Non-canonical num_bins={num_bins}. Canonical spec requires num_bins=8 (PDF §6.6).")

    # Only compute orientation on mass pixels
    mask_bool = mass_mask > 0

    # Extract gradients at mass locations
    gx = gradient_x[mask_bool]
    gy = gradient_y[mask_bool]
    gmag = gradient_mag[mask_bool]

    if len(gx) == 0 or gmag.sum() == 0:
        return 0.0

    # Compute gradient orientations: atan2 returns [-π, π]
    orientations = np.arctan2(gy, gx)

    # Shift to [0, 2π]
    orientations = orientations + np.pi

    # Discretize into bins
    bin_width = 2.0 * np.pi / num_bins
    bin_indices = np.floor(orientations / bin_width).astype(int)

    # Handle edge case: orientation exactly at 2π
    bin_indices = np.clip(bin_indices, 0, num_bins - 1)

    # Weight by gradient magnitude (stronger gradients contribute more)
    bin_weights = np.zeros(num_bins, dtype=np.float64)
    for i in range(len(bin_indices)):
        bin_weights[bin_indices[i]] += gmag[i]

    # Normalize to probabilities
    total_weight = bin_weights.sum()
    if total_weight == 0:
        return 0.0

    probabilities = bin_weights / total_weight

    # Shannon entropy
    H = 0.0
    for p in probabilities:
        if p > 0:
            H -= p * np.log2(p)

    # Maximum entropy
    H_max = np.log2(num_bins)

    if H_max == 0:
        return 1.0

    # Orientation stability (normalized inverse entropy)
    theta = 1.0 - (H / H_max)

    return float(theta)

def metric_d_s(mass_mask: np.ndarray) -> float:
    """
    Structural thickness:
    Compute distance transform on mass mask, skeletonize, and read average thickness along skeleton.
    Normalize by min(h,w) so it's scale-aware.
    """
    h, w = mass_mask.shape
    if mass_mask.sum() == 0:
        return 0.0

    mask = (mass_mask > 0)
    dt = distance_transform_edt(mask)
    sk = skeletonize(mask).astype(np.uint8)
    ys, xs = np.nonzero(sk)
    if len(xs) == 0:
        return 0.0

    # thickness at skeleton ~ 2*radius
    thick = 2.0 * dt[ys, xs]
    norm = float(min(h, w))
    return float(thick.mean() / (norm + EPS))

Cell 6 — One canonical entry point: image → mask → kernel

In [ ]:
def compute_kernel_metrics_from_bgr(img_bgr: np.ndarray) -> Tuple[Dict, Dict]:
    """
    Canonical device function.
    Takes standardized BGR image and returns:
      metrics dict + fields dict
    """
    gray = bgr_to_gray_float(img_bgr)
    gx, gy, gmag = sobel_gradients(gray)
    mass_mask = robust_threshold_mask(gmag)

    mass_frac = metric_mass_fraction(mass_mask)
    if mass_frac < MIN_MASS_FRAC:
        metrics = {
            "delta_x": 0.0,
            "delta_y": 0.0,
            "r_v": metric_r_v(gmag),
            "rho_r": 0.0,
            "mu": 0.0,
            "x_p": 0.0,
            "theta": 0.0,
            "d_s": 0.0,
            "mass_fraction": float(mass_frac),
            "valid": 0,
            "quality_note": f"mass_fraction={mass_frac:.6f} < MIN_MASS_FRAC={MIN_MASS_FRAC:.6f}"
        }
        fields = {"gray": gray, "gmag": gmag, "mass_mask": mass_mask}
        return metrics, fields

    metrics = {
        "delta_x": metric_delta_x(mass_mask),
        "delta_y": metric_delta_y(mass_mask),
        "r_v": metric_r_v(gmag),
        "rho_r": metric_rho_r(mass_mask),
        "mu": metric_mu(mass_mask),
        "x_p": metric_x_p(gmag),
        "theta": metric_theta(gx, gy, gmag, mass_mask),
        "d_s": metric_d_s(mass_mask),
        "mass_fraction": float(mass_frac),
        "gradient_floor_85": float(np.percentile(gmag.flatten(), 85.0)),
        "gradient_ceiling_97": float(np.percentile(gmag.flatten(), 97.0)),
        "r_v_threshold": R_V_ABSOLUTE_THRESHOLD,
        "valid": 1,
        "quality_note": ""
    }

    if mass_frac < WARN_MASS_FRAC:
        metrics["quality_note"] = (
            f"mass_fraction={mass_frac:.4f} in low-structure band "
            f"({MIN_MASS_FRAC:.3f}–{WARN_MASS_FRAC:.3f}); treat as low-confidence."
        )

    fields = {"gray": gray, "gmag": gmag, "mass_mask": mass_mask}
    return metrics, fields

def compute_kernel_metrics_from_path(path: str) -> Tuple[Dict, Dict, Dict]:
    img_bgr, meta = standardize_image_from_path(path)
    metrics, fields = compute_kernel_metrics_from_bgr(img_bgr)
    return metrics, fields, meta

Cell 7 — Upload handling: accept ZIP or multiple images

In [ ]:
from google.colab import files

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}

def is_image_file(name: str) -> bool:
    return os.path.splitext(name.lower())[1] in IMAGE_EXTS

def collect_paths_from_upload(upload_dir: str) -> List[str]:
    """
    Finds images in upload_dir (recursively) and returns sorted paths.
    """
    paths = []
    for root, _, fnames in os.walk(upload_dir):
        for fn in fnames:
            if is_image_file(fn):
                paths.append(os.path.join(root, fn))
    return sorted(paths)

def upload_images_or_zip(upload_dir: str = "/content/vtl_upload") -> List[str]:
    """
    User can upload either:
      - one ZIP
      - many image files
      - or a mix (ZIP + images)
    All extracted/collected into upload_dir.
    Returns list of image paths.
    """
    os.makedirs(upload_dir, exist_ok=True)

    up = files.upload()
    if not up:
        return []

    # Write all uploaded files
    uploaded_paths = []
    for name, data in up.items():
        out_path = os.path.join(upload_dir, name)
        with open(out_path, "wb") as f:
            f.write(data)
        uploaded_paths.append(out_path)

    # Extract any zips into upload_dir
    for p in uploaded_paths:
        if p.lower().endswith(".zip"):
            with zipfile.ZipFile(p, "r") as z:
                z.extractall(upload_dir)

    # Collect images recursively
    img_paths = collect_paths_from_upload(upload_dir)
    if len(img_paths) == 0:
        raise ValueError("No images found. Upload images or a ZIP containing images.")
    return img_paths

Cell 8 — Run batch (and show a “highlighted subset” option)

In [ ]:
def run_batch(image_paths: List[str],
              highlighted: Optional[List[str]] = None,
              save_csv_path: str = "/content/kernel_metrics_batch.csv") -> pd.DataFrame:
    """
    image_paths: all images to process
    highlighted: optional subset (list of basenames or full paths)
                 If provided, we add a boolean column `highlighted`.
    """
    highlighted_set = set()
    if highlighted:
        for h in highlighted:
            highlighted_set.add(os.path.basename(h))
            highlighted_set.add(h)

    rows = []
    for p in image_paths:
        metrics, _, meta = compute_kernel_metrics_from_path(p)
        row = {**meta, **metrics}
        row["filename"] = os.path.basename(p)
        row["highlighted"] = int((os.path.basename(p) in highlighted_set) or (p in highlighted_set))
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(save_csv_path, index=False)
    print(f"Saved: {save_csv_path}  (rows={len(df)})")
    return df

Cell 9 — Quick single image test (same pipeline)

In [ ]:
# Upload either one image or a ZIP; we'll process the first image as "single"
paths = upload_images_or_zip()
print(f"Found {len(paths)} images. First: {paths[0]}")

metrics, fields, meta = compute_kernel_metrics_from_path(paths[0])
print("Kernel Metrics:")
for k in ["delta_x","delta_y","r_v","rho_r","mu","x_p","theta","d_s","mass_fraction","valid"]:
    print(f"  {k}: {metrics[k]:.4f}")
if metrics.get("quality_note"):
    print("Note:", metrics["quality_note"])

Cell 10 — Batch run + optional highlight list + preview

In [ ]:
# Optional: specify highlight filenames (or full paths) you care about:
HIGHLIGHT = []  # e.g., ["img_001.png", "img_042.jpg"]

df = run_batch(paths, highlighted=HIGHLIGHT, save_csv_path="/content/kernel_metrics_batch.csv")
df.head(10)

Cell 11 — Visual sanity: show mask (device QA)

In [ ]:
import matplotlib.pyplot as plt

def show_mask_preview(fields: Dict, title: str = ""):
    gray = fields["gray"]
    gmag = fields["gmag"]
    mask = fields["mass_mask"]

    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1); plt.imshow(gray, cmap="gray"); plt.title("Gray"); plt.axis("off")
    plt.subplot(1,3,2); plt.imshow(gmag, cmap="gray"); plt.title("Grad Mag"); plt.axis("off")
    plt.subplot(1,3,3); plt.imshow(mask, cmap="gray"); plt.title("Mass Mask"); plt.axis("off")
    if title:
        plt.suptitle(title)
    plt.show()

show_mask_preview(fields, title=os.path.basename(meta["path"]))

Cell 12 - Two QA checks + mask_sha256

In [ ]:
# Cell 12 — Mask QA utilities (independent)
# Requires: numpy, cv2, skimage.measure.label

import hashlib
import numpy as np
from skimage.measure import label

def _as_bool_mask(mask: np.ndarray) -> np.ndarray:
    """Normalize incoming mask to boolean [H,W]."""
    if mask is None:
        raise ValueError("mass_mask is None")
    m = mask
    if m.dtype != np.bool_:
        # accept 0/1, 0/255, floats, etc.
        m = m.astype(np.float32)
        m = m > 0.5 if m.max() <= 1.0 else m > 0
    return m

def mask_sha256(mask_bool: np.ndarray) -> str:
    """Hash the binary mask bytes for determinism checks."""
    mb = np.ascontiguousarray(mask_bool.astype(np.uint8))
    return hashlib.sha256(mb.tobytes()).hexdigest()

def mask_qa(mask: np.ndarray,
            min_mass_frac: float = None,
            warn_mass_frac: float = None,
            max_mass_frac: float = 0.85,
            connectivity: int = 1):
    """
    Returns:
      qa = dict with coverage + component stats + status + reasons + mask_sha256
    Status:
      PASS / WARN / FAIL
    """
    m = _as_bool_mask(mask)
    mass_fraction = float(m.mean())

    # Pull your notebook constants if they exist
    if min_mass_frac is None:
        min_mass_frac = float(globals().get("MIN_MASS_FRAC", 0.001))
    if warn_mass_frac is None:
        warn_mass_frac = float(globals().get("WARN_MASS_FRAC", 0.03))

    status = "PASS"
    reasons = []

    # Coverage sanity
    if mass_fraction < min_mass_frac:
        status = "FAIL"
        reasons.append(f"mass_fraction<{min_mass_frac:g} (too little structure)")
    elif mass_fraction < warn_mass_frac:
        status = "WARN"
        reasons.append(f"mass_fraction<{warn_mass_frac:g} (weak signal / sparse mask)")

    if mass_fraction > max_mass_frac:
        status = "FAIL" if status == "PASS" else status
        reasons.append(f"mass_fraction>{max_mass_frac:g} (too much structure / edge-snow risk)")

    # Components sanity
    lab = label(m, connectivity=connectivity)
    n_components = int(lab.max())
    if n_components == 0:
        # covered by low mass fraction, but make explicit
        status = "FAIL"
        reasons.append("no connected components")

        largest_component_fraction = 0.0
    else:
        counts = np.bincount(lab.ravel())
        # counts[0] is background
        comp_counts = counts[1:]
        largest = int(comp_counts.max()) if comp_counts.size else 0
        total_on = int(m.sum())
        largest_component_fraction = float(largest / max(total_on, 1))

        # Heuristics for “all islands, no backbone”
        if n_components > 2000 and largest_component_fraction < 0.05:
            status = "WARN" if status == "PASS" else status
            reasons.append("many components + tiny largest component (island soup)")
        if n_components > 10000:
            status = "WARN" if status == "PASS" else status
            reasons.append("extremely high component count (likely texture-driven)")

    return {
        "mass_fraction": mass_fraction,
        "n_components": n_components,
        "largest_component_fraction": float(largest_component_fraction),
        "mask_sha256": mask_sha256(m),
        "mask_status": status,
        "mask_reasons": "; ".join(reasons) if reasons else ""
    }

qa = mask_qa(fields["mass_mask"])

print("=== MASK QA ===")
for k, v in qa.items():
    print(f"{k}: {v}")

Cell 13. Summary Report

In [ ]:
# Cell B — Batch report table + quick diagnostics
import os
import pandas as pd
import numpy as np

CSV_PATH = "/content/kernel_metrics_batch.csv"   # change if needed
OUT_DIR  = "/content/vtl_reports"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)

print("Rows:", len(df))
print("Columns:", list(df.columns))

# --- sanity: duplicates by sha256 (same image run twice, or same bytes) ---
dup_count = int(df.duplicated(subset=["sha256"]).sum()) if "sha256" in df.columns else 0
print("Duplicate sha256 rows:", dup_count)

# --- r_v vs mass_fraction comparison (diagnostic only) ---
if "r_v" in df.columns and "mass_fraction" in df.columns:
    print(f"r_v range: [{df['r_v'].min():.3f}, {df['r_v'].max():.3f}]")
    print(f"r_v mean: {df['r_v'].mean():.3f} ± {df['r_v'].std():.3f}")
    print(f"mass_fraction (percentile): {df['mass_fraction'].mean():.3f} ± {df['mass_fraction'].std():.6f}")
    print(f"Note: r_v uses absolute threshold, mass_fraction uses percentile (different methods)")

# --- summary stats for the kernel vector ---
core_cols = [c for c in ["delta_x","delta_y","r_v","rho_r","mu","x_p","theta","d_s","mass_fraction","valid"] if c in df.columns]
summary = df[core_cols].describe().T
summary_path = os.path.join(OUT_DIR, "kernel_summary.csv")
summary.to_csv(summary_path)
print("Saved:", summary_path)

# --- if your batch CSV includes mask QA columns, surface them ---
qa_cols = [c for c in ["mask_status","mask_reasons","n_components","largest_component_fraction","mask_sha256"] if c in df.columns]
if qa_cols:
    # show counts
    print("\nMask status counts:")
    print(df["mask_status"].value_counts(dropna=False))

    # list the worst offenders first
    offenders = df.copy()
    # Sort: FAIL first, then WARN, then by n_components descending if present
    status_rank = {"FAIL": 0, "WARN": 1, "PASS": 2}
    offenders["_rank"] = offenders["mask_status"].map(status_rank).fillna(9)
    sort_cols = ["_rank"]
    if "n_components" in offenders.columns:
        sort_cols += ["n_components"]
        offenders = offenders.sort_values(sort_cols, ascending=[True, False])
    else:
        offenders = offenders.sort_values(sort_cols, ascending=True)

    show_cols = ["filename","path"] + qa_cols + core_cols
    show_cols = [c for c in show_cols if c in offenders.columns]

    print("\nTop 15 WARN/FAIL:")
    print(offenders.loc[offenders["mask_status"].isin(["FAIL","WARN"]), show_cols].head(15).to_string(index=False))

    offenders_out = os.path.join(OUT_DIR, "kernel_with_mask_qa__sorted.csv")
    offenders.drop(columns=["_rank"], errors="ignore").to_csv(offenders_out, index=False)
    print("\nSaved:", offenders_out)
else:
    print("\nNote: batch CSV does not include mask QA columns yet.")
    print("If you want QA in the batch report, we can append QA fields during run_batch,")
    print("or post-hoc recompute QA per image by reloading fields['mass_mask'] for each path.")

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
df.head()
display(df)

Cell 14. r_v + Gradient Field Package Report

- Summary statistics for r_v, gradient_floor_85, tail_gap
- Correlation (r_v ↔ gradient_floor_85)
- Top 10 highest r_v (gradient-quiet / absent field)
- Bottom 10 lowest r_v (gradient-active / textured field)
- Interpretation guide (how to read the numbers)

In [ ]:
# ============================
# NEW CELL: r_v + Gradient Field Package Report
# Location: Insert after "Cell 13. Summary Report"
# Purpose: Generate per-image r_v interpretation with gradient context
# ============================

import os
import pandas as pd
import numpy as np

# Setup
os.makedirs("/content/vtl_reports", exist_ok=True)

BATCH_CSV = "/content/kernel_metrics_batch.csv"
OUT_MD = "/content/vtl_reports/rv_gradient_field_report.md"

# Load data
if not os.path.exists(BATCH_CSV):
    raise FileNotFoundError(f"Batch CSV not found: {BATCH_CSV}")

df = pd.read_csv(BATCH_CSV)

# Ensure required columns exist
required_cols = ['filename', 'r_v', 'gradient_floor_85', 'gradient_ceiling_97']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Calculate derived metrics
df['tail_gap'] = df['gradient_ceiling_97'] - df['gradient_floor_85']
df['efa'] = df['gradient_floor_85']  # Edge Field Activation (alias for clarity)

# Sort by r_v (descending) for report
df_sorted = df.sort_values('r_v', ascending=False).copy()

# ===========================
# TRUNCATED TABLE (Top 10 + Bottom 10)
# ===========================

print("=" * 80)
print("r_v + GRADIENT FIELD REPORT")
print("=" * 80)
print()

# Summary statistics
print("SUMMARY STATISTICS")
print("-" * 80)
print(f"Total images: {len(df)}")
print(f"\nr_v (Void Ratio):")
print(f"  Mean:  {df['r_v'].mean():.4f}")
print(f"  Std:   {df['r_v'].std():.4f}")
print(f"  Min:   {df['r_v'].min():.4f}")
print(f"  Max:   {df['r_v'].max():.4f}")
print(f"  Range: {df['r_v'].max() - df['r_v'].min():.4f}")

print(f"\ngradient_floor_85 (Baseline Activation / EFA):")
print(f"  Mean:  {df['gradient_floor_85'].mean():.4f}")
print(f"  Std:   {df['gradient_floor_85'].std():.4f}")
print(f"  Min:   {df['gradient_floor_85'].min():.4f}")
print(f"  Max:   {df['gradient_floor_85'].max():.4f}")

print(f"\ntail_gap (ceiling_97 - floor_85):")
print(f"  Mean:  {df['tail_gap'].mean():.4f}")
print(f"  Std:   {df['tail_gap'].std():.4f}")

# Correlation
corr = df['r_v'].corr(df['gradient_floor_85'])
print(f"\nCorrelation: r_v ↔ gradient_floor_85 = {corr:.4f}")

print("\n" + "=" * 80)
print("TRUNCATED TABLE (Top 10 Highest r_v + Bottom 10 Lowest r_v)")
print("=" * 80)
print()

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 40)

# Select columns for display
display_cols = ['filename', 'r_v', 'gradient_floor_85', 'gradient_ceiling_97', 'tail_gap']
if 'rho_r' in df.columns:
    display_cols.append('rho_r')
if 'delta_x' in df.columns:
    display_cols.append('delta_x')

# Top 10
print("TOP 10 (Highest r_v - Gradient-Quiet / Absent Field)")
print("-" * 80)
top10 = df_sorted.head(10)[display_cols]
print(top10.to_string(index=False))

print("\n" + "-" * 80)
print("BOTTOM 10 (Lowest r_v - Gradient-Active / Textured Field)")
print("-" * 80)
bottom10 = df_sorted.tail(10)[display_cols]
print(bottom10.to_string(index=False))

print("\n" + "=" * 80)

# ===========================
# INTERPRETATION GUIDE
# ===========================

print("\nINTERPRETATION GUIDE")
print("=" * 80)
print("""
r_v (Void Ratio / Gradient-Quiet Fraction):
  High r_v (>0.85): Gradient-quiet regions, smooth surfaces, absent field
  Low r_v (<0.65): Gradient-active regions, textured surfaces, present field

gradient_floor_85 (Edge Field Activation - EFA):
  High EFA (>0.3): Strong baseline gradients, texture everywhere
  Low EFA (<0.1): Weak baseline gradients, smooth/quiet field

tail_gap (ceiling_97 - floor_85):
  Large gap (>0.4): Quiet field + few extreme edges (isolated objects)
  Small gap (<0.2): Edge energy distributed uniformly (flat rendering)

Correlation: r_v ↔ gradient_floor_85 ≈ -0.97
  → r_v measures gradient sparsity, not compositional void
  → Texture elimination → high r_v (quiet field)
  → Texture presence → low r_v (active field)

Within-Engine Usage:
  ✓ Compare r_v across prompts/versions for same engine
  ✓ Detect rendering style changes (texture manipulation)
  ✓ Measure field activation independently of object packing

Cross-Engine Usage:
  ⚠ Always report r_v alongside gradient_floor_85 and gradient_ceiling_97
  ⚠ Different engines have different baseline EFA values
  ⚠ Interpret r_v differences in gradient field context
""")

# ===========================
# MARKDOWN REPORT
# ===========================

with open(OUT_MD, 'w') as f:
    f.write("# r_v + Gradient Field Report\n\n")
    f.write("**VTL Kernel Metrics - Field Activation Analysis**\n\n")
    f.write("---\n\n")

    # Summary
    f.write("## Summary Statistics\n\n")
    f.write(f"- **Total images**: {len(df)}\n")
    f.write(f"- **r_v**: {df['r_v'].mean():.4f} ± {df['r_v'].std():.4f} (range: [{df['r_v'].min():.4f}, {df['r_v'].max():.4f}])\n")
    f.write(f"- **gradient_floor_85 (EFA)**: {df['gradient_floor_85'].mean():.4f} ± {df['gradient_floor_85'].std():.4f}\n")
    f.write(f"- **tail_gap**: {df['tail_gap'].mean():.4f} ± {df['tail_gap'].std():.4f}\n")
    f.write(f"- **Correlation**: r_v ↔ gradient_floor_85 = **{corr:.4f}**\n\n")

    f.write("---\n\n")

    # Categorical breakdown
    f.write("## Field Activation Categories\n\n")

    # High r_v / Low EFA
    quiet_field = df[(df['r_v'] > 0.85) & (df['gradient_floor_85'] < 0.1)]
    f.write(f"### Gradient-Quiet Field (r_v > 0.85, EFA < 0.1): {len(quiet_field)} images\n")
    f.write("*Smooth surfaces, absent field, texture-eliminated*\n\n")
    if len(quiet_field) > 0:
        for _, row in quiet_field.head(5).iterrows():
            f.write(f"- `{row['filename']}`: r_v={row['r_v']:.3f}, floor={row['gradient_floor_85']:.3f}\n")
    f.write("\n")

    # Low r_v / High EFA
    active_field = df[(df['r_v'] < 0.65) & (df['gradient_floor_85'] > 0.3)]
    f.write(f"### Gradient-Active Field (r_v < 0.65, EFA > 0.3): {len(active_field)} images\n")
    f.write("*Textured surfaces, present field, high-frequency structure*\n\n")
    if len(active_field) > 0:
        for _, row in active_field.head(5).iterrows():
            f.write(f"- `{row['filename']}`: r_v={row['r_v']:.3f}, floor={row['gradient_floor_85']:.3f}\n")
    f.write("\n")

    # Moderate range
    moderate_field = df[(df['r_v'] >= 0.65) & (df['r_v'] <= 0.85)]
    f.write(f"### Moderate Field (0.65 ≤ r_v ≤ 0.85): {len(moderate_field)} images\n")
    f.write("*Mixed texture, moderate activation*\n\n")

    f.write("---\n\n")

    # Full table
    f.write("## Complete Dataset (sorted by r_v descending)\n\n")
    f.write("| Filename | r_v | floor_85 | ceiling_97 | tail_gap | Interpretation |\n")
    f.write("|----------|-----|----------|------------|----------|----------------|\n")

    for _, row in df_sorted.iterrows():
        filename = row['filename'][:35] + '...' if len(row['filename']) > 35 else row['filename']
        rv = row['r_v']
        floor = row['gradient_floor_85']
        ceiling = row['gradient_ceiling_97']
        gap = row['tail_gap']

        # Interpretation
        if rv > 0.85 and floor < 0.1:
            interp = "Quiet field"
        elif rv < 0.65 and floor > 0.3:
            interp = "Active field"
        elif gap > 0.4:
            interp = "Isolated edges"
        elif gap < 0.2:
            interp = "Flat rendering"
        else:
            interp = "Moderate"

        f.write(f"| {filename} | {rv:.3f} | {floor:.3f} | {ceiling:.3f} | {gap:.3f} | {interp} |\n")

    f.write("\n---\n\n")

    # Interpretation guide
    f.write("## Interpretation Guide\n\n")
    f.write("### r_v (Void Ratio / Gradient-Quiet Fraction)\n")
    f.write("- **High r_v (>0.85)**: Gradient-quiet regions, smooth surfaces, absent field\n")
    f.write("- **Low r_v (<0.65)**: Gradient-active regions, textured surfaces, present field\n")
    f.write("- **Measures**: Fraction of frame where gradient magnitude < τ (0.15)\n\n")

    f.write("### gradient_floor_85 (Edge Field Activation - EFA)\n")
    f.write("- **High EFA (>0.3)**: Strong baseline gradients, texture everywhere\n")
    f.write("- **Low EFA (<0.1)**: Weak baseline gradients, smooth/quiet field\n")
    f.write("- **Measures**: 85th percentile gradient magnitude (activation threshold)\n\n")

    f.write("### tail_gap (ceiling_97 - floor_85)\n")
    f.write("- **Large gap (>0.4)**: Quiet field + few extreme edges (isolated objects)\n")
    f.write("- **Small gap (<0.2)**: Edge energy distributed uniformly (flat rendering)\n")
    f.write("- **Measures**: Range between peak edges and baseline activation\n\n")

    f.write("### Key Insight: r_v ↔ gradient_floor_85 ≈ -0.97\n")
    f.write("r_v measures **gradient sparsity**, not compositional void.\n")
    f.write("- Texture elimination → high r_v (quiet field)\n")
    f.write("- Texture presence → low r_v (active field)\n\n")

    f.write("### Usage Guidelines\n")
    f.write("**Within-Engine**:\n")
    f.write("- ✓ Compare r_v across prompts/versions for same engine\n")
    f.write("- ✓ Detect rendering style changes (texture manipulation)\n")
    f.write("- ✓ Measure field activation independently of object packing\n\n")

    f.write("**Cross-Engine**:\n")
    f.write("- ⚠ Always report r_v alongside gradient_floor_85 and gradient_ceiling_97\n")
    f.write("- ⚠ Different engines have different baseline EFA values\n")
    f.write("- ⚠ Interpret r_v differences in gradient field context\n\n")

    f.write("---\n\n")
    f.write(f"*Report generated from: {BATCH_CSV}*\n")
    f.write(f"*Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}*\n")

print(f"\n✓ Markdown report saved: {OUT_MD}")
print("=" * 80)


In [ ]:
# ============================
# Quick CSV Export: r_v + Gradient Field Package
# Creates sortable CSV with interpretation column
# ============================

import os
import pandas as pd

BATCH_CSV = "/content/kernel_metrics_batch.csv"
OUT_CSV = "/content/vtl_reports/rv_gradient_package.csv"

# Load data
df = pd.read_csv(BATCH_CSV)

# Calculate derived metrics
df['tail_gap'] = df['gradient_ceiling_97'] - df['gradient_floor_85']

# Add interpretation column
def interpret_rv(row):
    rv = row['r_v']
    floor = row['gradient_floor_85']
    gap = row['tail_gap']

    if rv > 0.85 and floor < 0.1:
        return "Quiet field"
    elif rv < 0.65 and floor > 0.3:
        return "Active field"
    elif gap > 0.4:
        return "Isolated edges"
    elif gap < 0.2:
        return "Flat rendering"
    else:
        return "Moderate"

df['interpretation'] = df.apply(interpret_rv, axis=1)

# Select columns and sort by filename
export_cols = ['filename', 'r_v', 'gradient_floor_85', 'gradient_ceiling_97', 'tail_gap', 'rho_r', 'interpretation']
df_export = df[export_cols].sort_values('filename').copy()

# Round numeric columns for readability
df_export['r_v'] = df_export['r_v'].round(4)
df_export['gradient_floor_85'] = df_export['gradient_floor_85'].round(4)
df_export['gradient_ceiling_97'] = df_export['gradient_ceiling_97'].round(4)
df_export['tail_gap'] = df_export['tail_gap'].round(4)

# Save
df_export.to_csv(OUT_CSV, index=False)

print(f"✓ CSV exported: {OUT_CSV}")
print(f"  Rows: {len(df_export)}")
print(f"  Columns: {', '.join(export_cols)}")
print(f"  Sorted by: filename")
print()
print("Preview (first 5 rows):")
print(df_export.head().to_string(index=False))


QA SECTION

Cell A — Mask QA + determinism hash + optional saved preview + JSON

In [ ]:
# Cell A — Mask QA + determinism artifacts (independent)
# Requires: compute_kernel_metrics_from_path(path) -> (metrics:dict, fields:dict, meta:dict)
#           mask_qa(mask) -> dict   (if not present, a fallback implementation is included)

import os, json, hashlib
import numpy as np
import pandas as pd

IN_CSV = "/content/kernel_metrics_batch.csv"
OUT_DIR = "/content/vtl_mask_qa"
OUT_CSV = "/content/vtl_reports/kernel_metrics_batch_qa.csv"

SAVE_PREVIEWS = True  # set False if you don't want preview PNGs
EPS = 1e-9

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

assert os.path.exists(IN_CSV), f"Missing {IN_CSV}. Point IN_CSV at your batch CSV."

df = pd.read_csv(IN_CSV)

# --- Fallbacks (only used if your notebook doesn't already define these) ---
def _as_bool_mask(mask: np.ndarray) -> np.ndarray:
    if mask is None:
        raise ValueError("mass_mask is None")
    m = mask
    if m.dtype != np.bool_:
        m = m.astype(np.float32)
        m = (m > 0.5) if float(m.max()) <= 1.0 else (m > 0)
    return m

def _mask_sha256(mask_bool: np.ndarray) -> str:
    mb = np.ascontiguousarray(mask_bool.astype(np.uint8))
    return hashlib.sha256(mb.tobytes()).hexdigest()

def _fallback_mask_qa(mask: np.ndarray,
                      min_mass_frac: float = None,
                      warn_mass_frac: float = None,
                      max_mass_frac: float = 0.85,
                      connectivity: int = 1) -> dict:
    from skimage.measure import label

    m = _as_bool_mask(mask)
    mass_fraction = float(m.mean())

    if min_mass_frac is None:
        min_mass_frac = float(globals().get("MIN_MASS_FRAC", 0.001))
    if warn_mass_frac is None:
        warn_mass_frac = float(globals().get("WARN_MASS_FRAC", 0.03))

    status = "PASS"
    reasons = []

    if mass_fraction < min_mass_frac:
        status = "FAIL"
        reasons.append(f"mass_fraction<{min_mass_frac:g} (too little structure)")
    elif mass_fraction < warn_mass_frac:
        status = "WARN"
        reasons.append(f"mass_fraction<{warn_mass_frac:g} (weak signal / sparse mask)")

    if mass_fraction > max_mass_frac:
        status = "FAIL" if status == "PASS" else status
        reasons.append(f"mass_fraction>{max_mass_frac:g} (too much structure / edge-snow risk)")

    lab = label(m, connectivity=connectivity)
    n_components = int(lab.max())

    if n_components == 0:
        status = "FAIL"
        largest_frac = 0.0
        reasons.append("no connected components")
    else:
        counts = np.bincount(lab.ravel())
        comp_counts = counts[1:]
        largest = int(comp_counts.max()) if comp_counts.size else 0
        total_on = int(m.sum())
        largest_frac = float(largest / max(total_on, 1))

        # "island soup" heuristics
        if n_components > 2000 and largest_frac < 0.05:
            status = "WARN" if status == "PASS" else status
            reasons.append("many components + tiny largest component (island soup)")
        if n_components > 10000:
            status = "WARN" if status == "PASS" else status
            reasons.append("extremely high component count (likely texture-driven)")

    return {
        "mass_fraction": mass_fraction,
        "n_components": n_components,
        "largest_component_fraction": largest_frac,
        "mask_sha256": _mask_sha256(m),
        "mask_status": status,
        "mask_reasons": "; ".join(reasons) if reasons else ""
    }

# pick mask_qa implementation
mask_qa_fn = globals().get("mask_qa", None)
if mask_qa_fn is None:
    mask_qa_fn = _fallback_mask_qa

compute_fn = globals().get("compute_kernel_metrics_from_path", None)
assert compute_fn is not None, "Missing compute_kernel_metrics_from_path(path). Run your earlier cells first."

# --- Main loop ---
rows = []
print(f"Cell A: reading {IN_CSV} (rows={len(df)})")
for i, r in df.iterrows():
    path = r.get("path", None)
    if not isinstance(path, str) or not os.path.exists(path):
        rows.append({**r.to_dict(),
                     "qa_error": f"missing path or file not found: {path}"})
        continue

    try:
        metrics, fields, meta = compute_fn(path)
        qa = mask_qa_fn(fields.get("mass_mask"))

        # determinism artifact JSON (per-image)
        rec = {
            "path": path,
            "image_sha256": metrics.get("sha256", r.get("sha256", "")),
            "kernel_metrics": {k: float(metrics[k]) for k in ["delta_x","delta_y","r_v","rho_r","mu","x_p","theta","d_s","mass_fraction"] if k in metrics},
            "mask_qa": qa
        }

        sha = rec["image_sha256"] or os.path.basename(path)
        json_path = os.path.join(OUT_DIR, f"{sha}__maskqa.json")
        with open(json_path, "w") as f:
            json.dump(rec, f, indent=2)

        # optional preview save
        if SAVE_PREVIEWS:
            import matplotlib.pyplot as plt
            gray = fields.get("gray")
            gmag = fields.get("gmag")
            mmask = fields.get("mass_mask")
            if gray is not None and gmag is not None and mmask is not None:
                fig = plt.figure(figsize=(12,4))
                ax1 = plt.subplot(1,3,1); ax1.imshow(gray, cmap="gray"); ax1.set_title("Gray"); ax1.axis("off")
                ax2 = plt.subplot(1,3,2); ax2.imshow(gmag, cmap="gray"); ax2.set_title("Grad Mag"); ax2.axis("off")
                ax3 = plt.subplot(1,3,3); ax3.imshow(mmask, cmap="gray"); ax3.set_title("Mass Mask"); ax3.axis("off")
                plt.suptitle(os.path.basename(path))
                prev_path = os.path.join(OUT_DIR, f"{sha}__preview.png")
                plt.savefig(prev_path, dpi=160, bbox_inches="tight")
                plt.close(fig)

        out_row = r.to_dict()
        out_row.update({
            "mask_status": qa.get("mask_status",""),
            "mask_reasons": qa.get("mask_reasons",""),
            "mask_sha256": qa.get("mask_sha256",""),
            "n_components": qa.get("n_components", np.nan),
            "largest_component_fraction": qa.get("largest_component_fraction", np.nan),
            "qa_error": ""
        })
        rows.append(out_row)

        print(f"[{i+1}/{len(df)}] OK  mask={out_row['mask_status']:<4}  n={out_row['n_components']:<6}  largest={out_row['largest_component_fraction']:.4f}")

    except Exception as e:
        out_row = r.to_dict()
        out_row.update({"qa_error": repr(e)})
        rows.append(out_row)
        print(f"[{i+1}/{len(df)}] FAIL {repr(e)}")

out_df = pd.DataFrame(rows)
out_df.to_csv(OUT_CSV, index=False)
print(f"\nSaved QA-augmented CSV -> {OUT_CSV} (rows={len(out_df)})")
print(f"Artifacts folder -> {OUT_DIR}")

In [ ]:
# ============================
# Utility — Derive mask_mode from QA metrics
# ============================

import pandas as pd
import os

QA_CSV = "/content//vtl_reports/kernel_metrics_batch_qa.csv"  # adjust if needed

df = pd.read_csv(QA_CSV)

required = ["mask_status", "largest_component_fraction"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Cannot derive mask_mode, missing columns: {missing}")

def derive_mask_mode(row):
    if row["mask_status"] == "FAIL":
        return "INVALID"
    if row["largest_component_fraction"] >= 0.25:
        return "REGION_FIELD"
    return "TEXTURE_FIELD"

df["mask_mode"] = df.apply(derive_mask_mode, axis=1)

# Save back (overwrite or version — your choice)
OUT = QA_CSV
os.makedirs(os.path.dirname(OUT) or ".", exist_ok=True)
df.to_csv(OUT, index=False)

print("✅ mask_mode added to QA CSV")
print(df["mask_mode"].value_counts())

Cell B — Batch report table + determinism check + drift flags

In [ ]:
# Cell B — Batch report + determinism check + drift flags (schema-safe)

import os, numpy as np, pandas as pd

IN_CSV = "/content/vtl_reports/kernel_metrics_batch_qa.csv"   # output of Cell A
OUT_CSV = "/content/vtl_reports/kernel_report.csv"
EPS = 1e-9

os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
assert os.path.exists(IN_CSV), f"Missing {IN_CSV}. Run Cell A first."

df = pd.read_csv(IN_CSV)

# metrics we expect in your batch CSV
KERNEL_COLS = [c for c in ["delta_x","delta_y","r_v","rho_r","mu","x_p","theta","d_s","mass_fraction"] if c in df.columns]
BASE_COLS = [c for c in ["path","filename","sha256","h","w","max_side","valid","quality_note",
                         "mask_status","mask_reasons","mask_sha256","n_components","largest_component_fraction","qa_error"]
             if c in df.columns]

if "sha256" not in df.columns:
    raise ValueError("Your CSV has no sha256 column. Determinism check needs it.")

# Determinism grouping: same image sha256 appearing multiple times should yield same kernel vector
rows = []
for sha, g in df.groupby("sha256", dropna=False):
    g2 = g.copy()
    n_runs = len(g2)

    rec = {
        "sha256": sha,
        "n_runs": n_runs,
        "paths": " | ".join(g2["path"].astype(str).tolist()) if "path" in g2.columns else "",
        "any_valid": int(g2["valid"].max()) if "valid" in g2.columns else "",
        "any_qa_error": int((g2.get("qa_error","").astype(str) != "").any()),
    }

    # determinism diffs
    det_pass = True
    for c in KERNEL_COLS:
        vals = g2[c].astype(float).to_numpy()
        vmax = float(np.nanmax(vals))
        vmin = float(np.nanmin(vals))
        diff = vmax - vmin
        rec[f"{c}_max"] = vmax
        rec[f"{c}_min"] = vmin
        rec[f"{c}_diff"] = diff
        if diff > EPS:
            det_pass = False

    rec["deterministic_pass"] = int(det_pass)

    # mask status rollup
    if "mask_status" in g2.columns:
        # if any FAIL, mark FAIL; else if any WARN, mark WARN; else PASS
        statuses = set(g2["mask_status"].astype(str).tolist())
        if "FAIL" in statuses:
            rec["mask_rollup"] = "FAIL"
        elif "WARN" in statuses:
            rec["mask_rollup"] = "WARN"
        else:
            rec["mask_rollup"] = "PASS"

    rows.append(rec)

report = pd.DataFrame(rows)

# convenience flags
if len(KERNEL_COLS) > 0:
    report["any_kernel_drift"] = (report[[f"{c}_diff" for c in KERNEL_COLS]].max(axis=1) > EPS).astype(int)
else:
    report["any_kernel_drift"] = 0

report.to_csv(OUT_CSV, index=False)

print(f"Saved report -> {OUT_CSV} (rows={len(report)})")
print("\nReport head:")
print(report.head(10).to_string(index=False))

# quick summary
total = len(report)
multi = int((report["n_runs"] > 1).sum())
fails = int((report["deterministic_pass"] == 0).sum())
print(f"\nSummary: unique_images={total}  images_with_repeats={multi}  determinism_fails={fails}  eps={EPS:g}")

Additional Modes and Analysis

Cell 1 — Batch “Structure Typing” Summary (counts + rates)

In [ ]:
# ============================
# Cell 2 — Batch “Structure Typing” Summary (counts + rates)
# What: summarizes TEXTURE_FIELD vs REGION_FIELD vs INVALID across the batch QA CSV.
# Why: turns “lots of WARN” into a quantified profile.
# ============================

import os
import pandas as pd

QA_CSV = "/content/vtl_reports/kernel_metrics_batch_qa.csv"   # change if needed

df = pd.read_csv(QA_CSV)
needed = ["filename","mask_status","mask_mode","mass_fraction","n_components","largest_component_fraction"]
missing = [c for c in needed if c not in df.columns]
if missing:
  print("⚠️ Missing columns in QA CSV:", missing)
else:
  print("Rows:", len(df))
  print("\n== mask_mode counts ==")
  print(df["mask_mode"].value_counts(dropna=False))

  print("\n== mask_status counts ==")
  print(df["mask_status"].value_counts(dropna=False))

  # Rates
  mode_rate = (df["mask_mode"].value_counts(normalize=True) * 100).round(2)
  status_rate = (df["mask_status"].value_counts(normalize=True) * 100).round(2)

  print("\n== mask_mode rates (%) ==")
  print(mode_rate)

  print("\n== mask_status rates (%) ==")
  print(status_rate)

  print("\n== quick stats by mask_mode ==")
  g = df.groupby("mask_mode")[["mass_fraction","n_components","largest_component_fraction"]].agg(["median","mean","min","max"])
  print(g)


Cell 2 — Outlier Finder (top-N “most texture”, “most region”, “most sparse”, etc.)

In [ ]:
# ============================
# Cell 3 — Outlier Finder (top-N “most texture”, “most region”, “most sparse”, etc.)
# What: prints the most extreme rows for key QA signals + kernel metrics.
# Why: quickly see what kinds of images trigger each behavior.
# ============================

import pandas as pd

BATCH_CSV = "/content/kernel_metrics_batch.csv"
QA_CSV    = "//content/vtl_reports/kernel_metrics_batch_qa.csv"

m = pd.read_csv(BATCH_CSV)
q = pd.read_csv(QA_CSV)

# Merge on filename if present; fallback on path
key = "filename" if "filename" in m.columns and "filename" in q.columns else ("path" if "path" in m.columns and "path" in q.columns else None)
if key is None:
  raise ValueError("No common join key found between batch and QA (need filename or path).")

df = m.merge(q, on=key, how="left", suffixes=("","_qa"))

def show_top(col, n=10, asc=False):
  if col not in df.columns:
    print(f"⚠️ Missing col: {col}")
    return
  sub = df[[key, col]].dropna().sort_values(col, ascending=asc).head(n)
  print(f"\n== Top {n} {'LOW' if asc else 'HIGH'}: {col} ==")
  print(sub.to_string(index=False))

# Texture-ness proxies
show_top("n_components", n=10, asc=False)
show_top("largest_component_fraction", n=10, asc=True)

# Mask coverage extremes
show_top("mass_fraction", n=10, asc=True)
show_top("mass_fraction", n=10, asc=False)

# Kernel extremes (edit list as you like)
for col in ["delta_x","delta_y","r_v","rho_r","mu","x_p","theta","d_s"]:
  show_top(col, n=10, asc=False)

Cell 3 — Correlation Matrix (kernel-only, QA-only, and combined)

In [ ]:
# ============================
# Cell 4 — Correlation Matrix (kernel-only, QA-only, and combined)
# What: computes correlations for metrics and QA features, saves CSVs.
# Why: “researchy” sanity: are fields coupled? does TEXTURE_FIELD shift kernel behavior?
# ============================

import os
import pandas as pd

os.makedirs("/content/vtl_reports", exist_ok=True)

BATCH_CSV = "/content/kernel_metrics_batch.csv"
QA_CSV    = "/content/vtl_reports/kernel_metrics_batch_qa.csv"

m = pd.read_csv(BATCH_CSV)
q = pd.read_csv(QA_CSV)

key = "filename" if "filename" in m.columns and "filename" in q.columns else ("path" if "path" in m.columns and "path" in q.columns else None)
if key is None:
  raise ValueError("No common join key found between batch and QA (need filename or path).")

df = m.merge(q, on=key, how="left", suffixes=("","_qa"))

kernel_cols = [c for c in ["delta_x","delta_y","r_v","rho_r","mu","x_p","theta","d_s","mass_fraction"] if c in df.columns]
qa_cols     = [c for c in ["mass_fraction_qa","n_components","largest_component_fraction"] if c in df.columns]

# Some notebooks name QA mass_fraction the same; normalize:
if "mass_fraction_qa" not in df.columns and "mass_fraction" in q.columns:
  # if merge created duplicates, try pick the QA one
  if "mass_fraction_qa" in df.columns:
    pass
  else:
    # keep kernel mass_fraction; also include QA mass_fraction if it exists
    if "mass_fraction_y" in df.columns:
      df.rename(columns={"mass_fraction_y":"mass_fraction_qa","mass_fraction_x":"mass_fraction"}, inplace=True)
      qa_cols = [c for c in ["mass_fraction_qa","n_components","largest_component_fraction"] if c in df.columns]

def save_corr(cols, name):
  if len(cols) < 2:
    print(f"⚠️ Not enough columns for correlation: {name}")
    return None
  c = df[cols].corr(numeric_only=True)
  out = f"/content/vtl_reports/{name}.csv"
  c.to_csv(out)
  print("✅ Saved:", out)
  return c

c1 = save_corr(kernel_cols, "corr_kernel")
c2 = save_corr(qa_cols, "corr_maskqa")
c3 = save_corr(kernel_cols + qa_cols, "corr_combined")

# Quick peek (print a small slice if large)
if c3 is not None:
  print("\nCombined corr (rounded):")
  print(c3.round(3).to_string())

Cell 4 — Determinism Audit (find duplicates + compare vectors)

In [ ]:
# ============================
# Cell 5 — Determinism Audit (find duplicates + compare vectors)
# What: detects duplicated images (by sha256) and checks metric vector equality/tolerance.
# Why: catches “same image, different vector” drift instantly.
# ============================

import pandas as pd
import numpy as np

BATCH_CSV = "/content/kernel_metrics_batch.csv"
df = pd.read_csv(BATCH_CSV)

if "sha256" not in df.columns:
  raise ValueError("Batch CSV missing sha256 column.")

metric_cols = [c for c in ["delta_x","delta_y","r_v","rho_r","mu","x_p","theta","d_s","mass_fraction"] if c in df.columns]

dups = df[df.duplicated("sha256", keep=False)].sort_values("sha256")
print("Duplicate groups:", dups["sha256"].nunique())
if len(dups) == 0:
  print("✅ No duplicates by sha256.")
else:
  # For each group, compute max pairwise absolute deviation per metric
  rows = []
  for h, g in dups.groupby("sha256"):
    arr = g[metric_cols].to_numpy(dtype=float)
    # max abs deviation from the first row
    base = arr[0]
    dev = np.max(np.abs(arr - base), axis=0)
    rows.append({
      "sha256": h,
      "n": len(g),
      **{f"max_abs_dev__{metric_cols[i]}": float(dev[i]) for i in range(len(metric_cols))}
    })
  out = pd.DataFrame(rows).sort_values("n", ascending=False)
  print(out.head(20).to_string(index=False))
  print("\nTip: if any max_abs_dev is > ~1e-6 (float noise aside), you have real drift.")

Cell 5 — Mask Mode Contrast (Gradient vs Region-biased)

In [ ]:
# ============================
# Cell 6 — Mask Mode Contrast (Gradient vs Region-biased)
# What: generates a SECOND mask with a region bias (closing + area filter) without touching kernel mask.
# Why: lets you compare kernel vectors under different structural priors.
#
# Notes:
# - This does NOT replace your canonical mask.
# - It produces a “contrast mask” to understand how much your kernel depends on edge-field vs region-field.
# ============================

import numpy as np
from skimage.morphology import binary_closing, disk, remove_small_objects
from skimage.measure import label

def region_bias_mask(mass_mask_bool: np.ndarray,
                     close_radius: int = 2,
                     min_obj_frac: float = 0.0005):
  """
  mass_mask_bool: canonical boolean mask (edge/gradient-driven)
  close_radius: small closing to connect near-by edge fragments
  min_obj_frac: remove tiny islands relative to image area
  """
  m = mass_mask_bool.astype(bool)
  if close_radius > 0:
    m = binary_closing(m, footprint=disk(close_radius))
  min_size = int(min_obj_frac * m.size)
  if min_size > 0:
    m = remove_small_objects(m, min_size=min_size)
  return m

def compare_masks_and_metrics(img_path: str):
  # You must already have compute_kernel_metrics_from_path returning fields incl. 'mass_mask'
  metrics, fields, meta = compute_kernel_metrics_from_path(img_path)
  m0 = fields["mass_mask"].astype(bool)

  m1 = region_bias_mask(m0, close_radius=2, min_obj_frac=0.0005)

  qa0 = mask_qa(m0)
  qa1 = mask_qa(m1)

  print("== Canonical mask QA ==")
  for k,v in qa0.items(): print(f"{k}: {v}")

  print("\n== Region-biased contrast mask QA ==")
  for k,v in qa1.items(): print(f"{k}: {v}")

  # Optional: if you have a way to recompute kernel metrics from a provided mask,
  # you can add that here. If not, at least visualize.
  return metrics, fields, meta, m0, m1

# Example:
# metrics, fields, meta, m0, m1 = compare_masks_and_metrics("/content/vtl_upload/0_0.png")

Cell 6 — “Drift Flags” Heuristic Layer (interpretive, not gating)

In [ ]:
# ============================
# Cell 8 — “Drift Flags” Heuristic Layer (interpretive, not gating)
# What: adds a few columns to the report indicating likely structural regime / risk of over-texture.
# Why: lets you filter and stratify without changing kernel math.
# ============================

import pandas as pd
import numpy as np
import os

IN_CSV  = "/content/vtl_reports/kernel_metrics_batch_qa.csv"
OUT_CSV = "/content/vtl_reports/kernel_metrics_batch_qa__flags.csv"
os.makedirs("/content/vtl_reports", exist_ok=True)

df = pd.read_csv(IN_CSV)

# Safe defaults; tune later
COMP_HI = 20000
LCF_LO  = 0.01

df["flag_island_soup"] = (df.get("n_components", 0) > COMP_HI) & (df.get("largest_component_fraction", 1.0) < LCF_LO)
df["flag_low_mass"]    = (df.get("mass_fraction", 1.0) < 0.01)
df["flag_high_mass"]   = (df.get("mass_fraction", 0.0) > 0.40)

# Optional: structure regime
def regime(row):
  lcf = row.get("largest_component_fraction", np.nan)
  nc  = row.get("n_components", np.nan)
  if pd.isna(lcf) or pd.isna(nc): return "unknown"
  if lcf > 0.25: return "region"
  if nc > COMP_HI and lcf < LCF_LO: return "texture"
  return "hybrid"

df["structure_regime"] = df.apply(regime, axis=1)

df.to_csv(OUT_CSV, index=False)
print("✅ Saved:", OUT_CSV)
print(df[["filename","mask_mode","mask_status","structure_regime","flag_island_soup","n_components","largest_component_fraction","mass_fraction"]].head(20).to_string(index=False))

Cell 7 — “One Row Per Image” Markdown Summary (auto-write a report.md)

In [ ]:
# ============================
# Cell 9 — “One Row Per Image” Markdown Summary (auto-write a report.md)
# What: generates a human-readable markdown file with the key numbers + structure typing.
# Why: makes the notebook output shareable without screenshots.
# ============================

import os
import pandas as pd

os.makedirs("/content/vtl_reports", exist_ok=True)

BATCH_CSV = "/content/kernel_metrics_batch.csv"
QA_CSV    = "/content/vtl_reports/kernel_metrics_batch_qa.csv"
OUT_MD    = "/content/vtl_reports/kernel_run_report.md"

m = pd.read_csv(BATCH_CSV)
q = pd.read_csv(QA_CSV)

key = "filename" if "filename" in m.columns and "filename" in q.columns else ("path" if "path" in m.columns and "path" in q.columns else None)
if key is None:
  raise ValueError("No common join key found between batch and QA (need filename or path).")

df = m.merge(q, on=key, how="left", suffixes=("","_qa"))

cols = [c for c in [
  key, "sha256",
  "delta_x","delta_y","r_v","rho_r","mu","x_p","theta","d_s","mass_fraction",
  "mask_mode","mask_status","mask_reasons","n_components","largest_component_fraction"
] if c in df.columns]

lines = []
lines.append("# VTL Kernel Run Report\n")
lines.append(f"- Images: {len(df)}\n")
if "mask_mode" in df.columns:
  lines.append("## Mask Modes\n")
  lines.append(df["mask_mode"].value_counts().to_string() + "\n\n")
if "mask_status" in df.columns:
  lines.append("## Mask Status\n")
  lines.append(df["mask_status"].value_counts().to_string() + "\n\n")

lines.append("## Per-image summary\n")
for _, r in df[cols].iterrows():
  name = r.get(key, "unknown")
  lines.append(f"### {name}\n")
  for c in cols:
    v = r.get(c, "")
    if pd.isna(v): v = ""
    lines.append(f"- **{c}**: {v}")
  lines.append("")

with open(OUT_MD, "w") as f:
  f.write("\n".join(lines))

print("✅ Wrote:", OUT_MD)

Appendix: Print tables and more

Cell 1 — Kernel Tables (read-only diagnostics)

In [ ]:
# Cell C — Kernel Tables (read-only diagnostics)
# Purpose: break out interpretable tables from kernel_metrics_batch.csv
# Does NOT write files. Safe to re-run.

import pandas as pd
import numpy as np

CSV_PATH = "/content/kernel_metrics_batch.csv"
df = pd.read_csv(CSV_PATH)

print("Rows:", len(df))
print("Columns:", list(df.columns))

# -----------------------------
# Table 1 — Core kernel vectors
# -----------------------------
core_cols = ["filename","delta_x","delta_y","r_v","rho_r","mu","x_p","theta","d_s","mass_fraction"]
core_cols = [c for c in core_cols if c in df.columns]

print("\n=== TABLE 1: Core Kernel Vectors (first 10) ===")
display(df[core_cols].head(10))

# -----------------------------
# Table 2 — Distribution summary
# -----------------------------
print("\n=== TABLE 2: Kernel Distribution Summary ===")
summary = df[[c for c in core_cols if c != "filename"]].describe().T
display(summary)

# -----------------------------
# Table 3 — Extremes (sanity)
# -----------------------------
def show_extremes(col, k=5):
    if col not in df.columns:
        return None
    return pd.concat([
        df.nsmallest(k, col)[["filename", col]],
        df.nlargest(k, col)[["filename", col]]
    ])

extreme_cols = ["r_v", "rho_r", "mass_fraction", "x_p", "theta"]

for c in extreme_cols:
    tbl = show_extremes(c)
    if tbl is not None:
        print(f"\n=== TABLE 3: Extremes for {c} ===")
        display(tbl)

# -----------------------------
# Table 4 — Determinism check
# -----------------------------
if "sha256" in df.columns:
    dup = df.groupby("sha256").size().reset_index(name="count")
    dup = dup[dup["count"] > 1]
    print("\n=== TABLE 4: Determinism Check (same image hashes) ===")
    if len(dup) == 0:
        print("✔ No duplicate sha256 rows detected.")
    else:
        display(dup)

# -----------------------------
# Table 5 — Simple correlations
# -----------------------------
corr_cols = [c for c in ["r_v","mass_fraction","rho_r","x_p","mu","theta","d_s"] if c in df.columns]
corr = df[corr_cols].corr()

print("\n=== TABLE 5: Kernel Correlation Matrix ===")
display(corr.round(3))

In [ ]:
# Cell — Paths + folders (Colab-safe)
import os

ROOT = "/content"
UPLOAD_DIR  = os.path.join(ROOT, "vtl_upload")
REPORT_DIR  = os.path.join(ROOT, "vtl_reports")
MASKQA_DIR  = os.path.join(ROOT, "vtl_mask_qa")

for d in [UPLOAD_DIR, REPORT_DIR, MASKQA_DIR]:
    os.makedirs(d, exist_ok=True)

print("OK folders:")
print("UPLOAD_DIR:", UPLOAD_DIR)
print("REPORT_DIR:", REPORT_DIR)
print("MASKQA_DIR:", MASKQA_DIR)

2 Cell — Mask mode contrast (canonical vs region-biased) -> stability table

In [ ]:
# ============================================
# Cell — Mask mode contrast (canonical vs region-biased) -> stability table
# What: compares kernel vector stability when the SAME image is masked two ways:
#   (1) Canonical mask (your current gradient-field threshold mask)
#   (2) Region-biased mask (component-filtered + lightly regularized mask)
# Why: lets you say: "kernel metrics are stable across structural regimes" (or not)
# Output:
#   - prints a table sorted by largest kernel drift
#   - saves CSV: /content/vtl_reports/mask_mode_contrast__canonical_vs_region.csv
# ============================================

import os, glob, hashlib
import numpy as np
import pandas as pd
import cv2
import inspect
import numpy as np

def _expects_float(fn) -> bool:
    """Heuristic: does fn's first param look like a scalar mass_fraction?"""
    try:
        sig = inspect.signature(fn)
        params = list(sig.parameters.values())
        if not params:
            return False
        p0 = params[0]
        ann = p0.annotation
        # annotation-based hint
        if ann in (float, int):
            return True
        # name-based hint
        if isinstance(p0.name, str) and "mass" in p0.name and "frac" in p0.name:
            return True
        return False
    except Exception:
        return False

# Optional deps (we’ll degrade gracefully if missing)
try:
    from skimage.measure import label
    from skimage.morphology import binary_closing, binary_opening, disk
    _HAS_SKIMAGE = True
except Exception:
    _HAS_SKIMAGE = False

# -----------------------------
# Config
# -----------------------------
UPLOAD_DIR = globals().get("UPLOAD_DIR", "/content/vtl_upload")
REPORT_DIR = globals().get("REPORT_DIR", "/content/vtl_reports")
os.makedirs(REPORT_DIR, exist_ok=True)

# Your canonical constants (fall back to notebook defaults if present)
GRAD_LOW_PCT  = float(globals().get("GRAD_LOW_PCT", 85.0))
GRAD_HIGH_PCT = float(globals().get("GRAD_HIGH_PCT", 97.0))
EDGE_MARGIN_PX = int(globals().get("EDGE_MARGIN_PX", 2))

# Region-biased tuning knobs (safe defaults; adjust if needed)
# Goal: suppress "island soup" by removing tiny components + mild closing/opening
REGION_MIN_AREA_PX = int(globals().get("REGION_MIN_AREA_PX", 64))   # drop very tiny comps
REGION_KEEP_TOPK   = int(globals().get("REGION_KEEP_TOPK", 250))    # or keep top-K largest comps
REGION_MORPH_R     = int(globals().get("REGION_MORPH_R", 1))        # disk radius for morph ops

CORE_KEYS = ["delta_x","delta_y","r_v","rho_r","mu","x_p","theta","d_s","mass_fraction","valid"]

# -----------------------------
# Hard requirements: your functions must exist
# -----------------------------
required = ["sobel_gradients", "robust_threshold_mask",
            "metric_delta_x","metric_r_v","metric_rho_r","metric_mu","metric_x_p","metric_theta","metric_d_s","metric_mass_fraction"]
missing = [n for n in required if n not in globals()]
if missing:
    raise RuntimeError(
        "Missing required function(s) from earlier cells: " + ", ".join(missing) +
        "\nMake sure you ran the main kernel code cells before this one."
    )

def _sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _read_gray_u8(path: str) -> np.ndarray:
    im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if im is None:
        raise ValueError(f"Could not read image: {path}")
    return im

def _as_bool(mask: np.ndarray) -> np.ndarray:
    m = mask
    if m.dtype != np.bool_:
        m = m.astype(np.float32)
        m = (m > 0.5) if m.max() <= 1.0 else (m > 0)
    return m.astype(bool)

def mass_mask_canonical(gray_u8: np.ndarray) -> np.ndarray:
    # canonical mask from your pipeline: gmag -> robust threshold mask
    _, _, gmag = sobel_gradients(gray_u8)
    m = robust_threshold_mask(gmag, GRAD_LOW_PCT, GRAD_HIGH_PCT, EDGE_MARGIN_PX)
    return _as_bool(m)

def mass_mask_region_biased(gray_u8: np.ndarray) -> np.ndarray:
    """
    Region-biased variant:
      - start from canonical mask (so you're still measuring the same "field")
      - remove tiny connected components (island soup suppression)
      - optionally keep only top-K largest components
      - mild closing/opening to merge speckle into region shapes
    """
    m = mass_mask_canonical(gray_u8)

    if not _HAS_SKIMAGE:
        # No skimage: basic fallback — just return canonical
        return m

    lab = label(m, connectivity=1)
    n = int(lab.max())
    if n <= 0:
        return m

    counts = np.bincount(lab.ravel())
    # counts[0] is background
    comp_sizes = counts[1:]
    comp_ids = np.arange(1, n+1)

    # Filter tiny components
    keep = comp_ids[comp_sizes >= REGION_MIN_AREA_PX]

    # If still too many, keep only top-K largest
    if keep.size > REGION_KEEP_TOPK:
        order = np.argsort(comp_sizes)[::-1]   # desc
        keep = comp_ids[order[:REGION_KEEP_TOPK]]

    keep_set = set(int(x) for x in keep.tolist())
    m2 = np.isin(lab, list(keep_set))

    # Mild morphology to stabilize regions
    if REGION_MORPH_R > 0:
        se = disk(REGION_MORPH_R)
        m2 = binary_closing(m2, se)
        m2 = binary_opening(m2, se)

    return m2.astype(bool)

def compute_kernel_from_gray_and_mask(gray_u8: np.ndarray, mask_bool: np.ndarray) -> dict:
    """
    Standalone kernel compute that adapts to your metric function signatures.

    CRITICAL: metric_theta and metric_x_p now require gradients (after canonical update).
    We must recompute gradients from gray to pass to these functions.
    """
    m = mask_bool.astype(bool)

    # Recompute gradients from grayscale (needed for theta and x_p)
    # Convert gray_u8 to float32 [0,1] if it's uint8
    if gray_u8.dtype == np.uint8:
        gray_float = gray_u8.astype(np.float32) / 255.0
    else:
        gray_float = gray_u8.astype(np.float32)

    # Compute Sobel gradients (same as in main pipeline)
    gx = cv2.Sobel(gray_float, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_float, cv2.CV_32F, 0, 1, ksize=3)
    gmag = np.sqrt(gx * gx + gy * gy)

    # Compute mass_fraction scalar
    mf = float(metric_mass_fraction(m))

    out = {}

    # Metrics that use mask only
    out["delta_x"] = float(metric_delta_x(m))
    out["delta_y"] = float(metric_delta_y(m))
    out["r_v"] = float(metric_r_v(gmag))
    out["rho_r"] = float(metric_rho_r(m))
    out["mu"] = float(metric_mu(m))
    out["d_s"] = float(metric_d_s(m))

    # Metrics that now require gradients (post-canonical update)
    out["x_p"] = float(metric_x_p(gmag))  # ✅ Pass gradient magnitude
    out["theta"] = float(metric_theta(gx, gy, gmag, m))  # ✅ Pass all gradients + mask

    out["mass_fraction"] = mf
    out["valid"] = 1.0 if mf > 0 else 0.0

    return out

# -----------------------------
# Run contrast
# -----------------------------
paths = sorted([
    os.path.join(UPLOAD_DIR, f) for f in os.listdir(UPLOAD_DIR)
    if f.lower().endswith((".png",".jpg",".jpeg",".webp"))
])
if not paths:
    raise RuntimeError(f"No images found in {UPLOAD_DIR}")

rows = []
for p in paths:
    gray = _read_gray_u8(p)

    m_can = mass_mask_canonical(gray)
    m_reg = mass_mask_region_biased(gray)

    k_can = compute_kernel_from_gray_and_mask(gray, m_can)
    k_reg = compute_kernel_from_gray_and_mask(gray, m_reg)

    # drift (region - canonical) and max abs drift
    d = {k: float(k_reg.get(k, np.nan) - k_can.get(k, np.nan)) for k in CORE_KEYS if k in k_can}
    max_abs = float(np.nanmax(np.abs(np.array(list(d.values()), dtype=np.float64))))

    rows.append({
        "filename": os.path.basename(p),
        "path": p,
        "img_sha256": _sha256_file(p),
        "mass_fraction_can": float(m_can.mean()),
        "mass_fraction_reg": float(m_reg.mean()),
        "max_abs_delta_kernel": max_abs,
        **{f"d_{k}": v for k, v in d.items()},
    })

df = pd.DataFrame(rows).sort_values("max_abs_delta_kernel", ascending=False)

df["dominant_drift_metric"] = df[
    [c for c in df.columns if c.startswith("d_") and c != "d_valid"]
].abs().idxmax(axis=1)

print("Mask mode contrast table (largest kernel drift first):")
display(df.head(30))

out_csv = os.path.join(REPORT_DIR, "mask_mode_contrast__canonical_vs_region.csv")
df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

print("\nQuick read:")
print("  If max_abs_delta_kernel is near ~0 for most rows => kernel is mask-regime stable.")
print("  If drift is large in specific metrics (e.g., d_r_v, d_rho_r) => those metrics are mask-sensitive under texture vs region structure.")
print("\nNote: If skimage is unavailable, region-biased falls back to canonical (drift will be ~0).")

Mask-mode contrast reveals that kernel metrics tied to geometric placement (Δx, θ, xₚ, dₛ) remain stable across texture- and region-dominant regimes, while density-coupled measures (ρᵣ, rᵥ) exhibit significant drift. This indicates that the kernel is structurally consistent, with regime sensitivity localized to metrics that explicitly encode material definition rather than spatial intent — a desired and interpretable behavior..

ρᵣ and rᵥ are mask-conditional metrics: canonical values quantify density/void under an energy-defined material field (texture-sensitive), while region-biased values quantify density/void under coherent connected mass (object-sensitive). Divergence between the two is reported as regime sensitivity, not error.

Canonical (gradient-defined / texture-sensitive mask)

Use this when you want field behavior:
- ρᵣ answers: “How densely does visual energy (edges/contrast transitions) pack, radially, across the frame?”
- rᵥ answers: “How much of the frame is not claimed by gradient energy (void relative to energy-defined material)?”

This is the right regime for:

- generative-model “spatial priors” work (because the model often expresses intent via texture/edge structure even when objects are ambiguous)

- failure/refusal artifacts (banding, stipple, pseudo-detail)
- “busy” images where the model is doing lots of micro-commitment

Downside: it will call texture “material” even when humans would call it “surface noise.”

Region-biased (connected/region-dominant mask)
- Use this when you want object-like material:
- ρᵣ answers: “How densely does coherent mass pack radially?”
- rᵥ answers: “How much frame is void relative to connected material regions?”

This is the right regime for:
- clean silhouettes / strong subject isolation
- compositional mass placement claims (big forms, not microtexture)
- comparisons where you want invariance against texture style (photographic grain, painterly hatch)

Downside: it can undercount meaningful fine structure (filigree, hair, foliage, lace, dense linework) and collapse it into “less material than it feels.”

3 Cell — Correlations + outliers by mask_status (TEXTURE_FIELD / REGION_FIELD / INVALID)

In [ ]:
# Cell — Correlations + outliers by mask_status (TEXTURE_FIELD / REGION_FIELD / INVALID)
import os
import numpy as np
import pandas as pd

QA_PATH = os.path.join(ROOT, "/content/vtl_reports/kernel_metrics_batch_qa.csv")  # change if needed

if not os.path.exists(QA_PATH):
    raise FileNotFoundError(f"Missing QA CSV: {QA_PATH}")

df = pd.read_csv(QA_PATH)

# Try to infer regime column name
reg_col = None
for c in ["mask_mode","structure_mode","regime","field_type"]:
    if c in df.columns:
        reg_col = c
        break

if reg_col is None:
    # fallback: use mask_status mapping if that's all you have
    if "mask_status" not in df.columns:
        raise RuntimeError("QA CSV has no mask_mode and no mask_status; can't split regimes.")
    reg_col = "mask_status"

num_cols = [c for c in ["delta_x","delta_y","r_v","rho_r","mu","x_p","theta","d_s","mass_fraction",
                        "n_components","largest_component_fraction"] if c in df.columns]

if len(num_cols) < 4:
    raise RuntimeError("Not enough numeric columns found for correlation analysis: " + str(num_cols))

def corr_block(sub: pd.DataFrame) -> pd.DataFrame:
    c = sub[num_cols].corr(numeric_only=True)
    return c

print("Regime column:", reg_col)
for g, sub in df.groupby(reg_col):
    if len(sub) < 5:
        continue
    print(f"\n=== {g} (n={len(sub)}) ===")
    display(corr_block(sub))

# Outliers: top 20 most extreme (|z|) within each regime for a chosen metric
METRIC = "n_components" if "n_components" in df.columns else "mass_fraction"
print(f"\nOutliers by regime for: {METRIC}")

out_rows = []
for g, sub in df.groupby(reg_col):
    if len(sub) < 10:
        continue
    x = sub[METRIC].astype(float)
    z = (x - x.mean()) / (x.std(ddof=0) + 1e-12)
    worst = sub.assign(z=z.abs()).sort_values("z", ascending=False).head(20)
    worst = worst[[c for c in ["filename","path",METRIC,"z","mask_status","mask_reasons"] if c in worst.columns]]
    worst.insert(0, "regime", g)
    out_rows.append(worst)

out = pd.concat(out_rows, ignore_index=True) if out_rows else pd.DataFrame()
display(out)
out_path = os.path.join(REPORT_DIR, "correlation_outliers_by_regime.csv")
out.to_csv(out_path, index=False)
print("Saved:", out_path)

4 Cell — Threshold sweep curve: WARN rate vs (low_pct, high_pct)

In [ ]:
# Cell — Threshold sweep curve: WARN rate vs (low_pct, high_pct)
import os
import numpy as np
import pandas as pd
import cv2

if "mask_qa" not in globals():
    raise RuntimeError("mask_qa(mask) is missing. Define it first (your QA utility cell).")

PATHS = sorted([os.path.join(UPLOAD_DIR, f) for f in os.listdir(UPLOAD_DIR)
                if f.lower().endswith((".png",".jpg",".jpeg",".webp"))])
if not PATHS:
    raise RuntimeError(f"No images found in {UPLOAD_DIR}")

def gray_u8(path):
    im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if im is None:
        raise ValueError(f"Bad image: {path}")
    return im

# Choose a grid (keep modest so it runs fast)
LOW_PCTS  = [70, 75, 80, 85, 90]
HIGH_PCTS = [92, 94, 96, 97, 98]

rows = []
for lo in LOW_PCTS:
    for hi in HIGH_PCTS:
        if hi <= lo:
            continue

        warn = 0
        fail = 0
        for p in PATHS:
            g = gray_u8(p)
            # canonical-ish (texture sensitive)
            gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
            gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
            mag = cv2.magnitude(gx, gy)

            t = 0.5*(np.percentile(mag, lo) + np.percentile(mag, hi))
            m = (mag >= t).astype(np.uint8)

            qa = mask_qa(m)
            if qa["mask_status"] == "WARN":
                warn += 1
            elif qa["mask_status"] == "FAIL":
                fail += 1

        n = len(PATHS)
        rows.append({
            "grad_low_pct": lo,
            "grad_high_pct": hi,
            "n": n,
            "warn_rate": warn / n,
            "fail_rate": fail / n,
            "pass_rate": 1.0 - (warn + fail) / n
        })

sweep = pd.DataFrame(rows).sort_values(["grad_low_pct","grad_high_pct"])
display(sweep)

out_path = os.path.join(REPORT_DIR, "threshold_sweep_warn_curve.csv")
sweep.to_csv(out_path, index=False)
print("Saved:", out_path)

5 Cell — Per-image fingerprint table (img_sha256 + mask_sha256 + kernel vector hash)

In [ ]:
# Cell — Per-image fingerprint table (img_sha256 + mask_sha256 + kernel vector hash)
import os, json, hashlib
import pandas as pd
import numpy as np

CSV_PATH = os.path.join(ROOT, "kernel_metrics_batch.csv")  # change if needed
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Missing batch CSV: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)

need = ["filename","sha256","delta_x","delta_y","r_v","rho_r","mu","x_p","theta","d_s","mass_fraction"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError("Missing columns in batch CSV: " + str(missing))

def kernel_vec_hash(row) -> str:
    vec = [float(row[c]) for c in ["delta_x","r_v","rho_r","mu","x_p","theta","d_s","mass_fraction"]]
    b = json.dumps(vec, separators=(",",":"), ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(b).hexdigest()

df["kernel_vec_sha256"] = df.apply(kernel_vec_hash, axis=1)

# If QA fields exist, keep them; otherwise just show the kernel fingerprint
keep = [c for c in ["filename","sha256","kernel_vec_sha256",
                    "mass_fraction","r_v","x_p","mu",
                    "mask_sha256","mask_status","mask_reasons"] if c in df.columns]

finger = df[keep].copy()
display(finger)

out_path = os.path.join(REPORT_DIR, "per_image_fingerprints.csv")
finger.to_csv(out_path, index=False)
print("Saved:", out_path)

6 Cell — Batch stability test (shuffle order, compare summary stats)

In [ ]:
# Cell — Batch stability test (shuffle order, compare summary stats)
import os
import numpy as np
import pandas as pd

CSV_PATH = os.path.join(ROOT, "kernel_metrics_batch.csv")  # change if needed
df = pd.read_csv(CSV_PATH)

cols = [c for c in ["delta_x","delta_y","r_v","rho_r","mu","x_p","theta","d_s","mass_fraction","valid"] if c in df.columns]
if not cols:
    raise RuntimeError("No kernel numeric columns found to test stability.")

def summary(d: pd.DataFrame) -> pd.Series:
    s = d[cols].describe().loc[["mean","std","min","max"]].stack()
    s.index = [f"{a}__{b}" for a,b in s.index]
    return s

base = summary(df)

# shuffle multiple times
K = 5
rows = []
for i in range(K):
    shuf = df.sample(frac=1.0, replace=False, random_state=1234+i).reset_index(drop=True)
    s = summary(shuf)
    diff = (s - base).abs()
    rows.append({
        "trial": i,
        "max_abs_diff": float(diff.max()),
        "mean_abs_diff": float(diff.mean()),
    })

stab = pd.DataFrame(rows)
display(stab)

out_path = os.path.join(REPORT_DIR, "batch_shuffle_stability.csv")
stab.to_csv(out_path, index=False)
print("Saved:", out_path)

# 📊 Interpretation Guidelines: VTL as a Coordinate System

*Read this section to understand how to interpret kernel values and avoid common misinterpretations of VTL measurements.*

---

## **What VTL Measures (And What It Doesn't)**

VTL Kernel Metrics is **not** a quality assessment tool. It does not measure:
- ❌ Aesthetic value
- ❌ Compositional "correctness"
- ❌ Image quality or fidelity

Instead, it functions as a **stylistic seismograph** - a measurement device that reveals **where and how compositional choices concentrate visual mass**, regardless of whether those choices are "good" or "bad."

This coordinate-system interpretation follows established precedent in perceptual measurement: just as color spaces (CIE L\*a\*b\*, HSV) locate colors without ranking them aesthetically, and the Weber-Fechner law describes sensory intensity without prescribing optimal levels, VTL locates compositions structurally before any aesthetic judgment is applied.

---

## **The Center Is Not The Problem**

### **Common Misinterpretation:**
> "Center-weighted composition = AI prior = collapse = failure"

### **Correct Interpretation:**
> "Center-weighted composition = **one valid coordinate in compositional space**"

**The issue is not that AI models use the center.** The issue is when models use **only** the center, **regardless of prompt**, creating compositional monoculture where diverse semantic content maps to identical spatial arrangements.

**Key distinction:**
- A human photographer choosing center-weighted composition for a portrait = **authorship**
- An AI model defaulting to center-weighted composition for every prompt = **collapse**

**VTL detects the difference by measuring the diversity of compositional coordinates across a dataset, not the coordinates themselves.**

---

## **No Single Score Equals Collapse**

**Collapse is not about individual values - it's about cluster tightness across a dataset.**

### **What Collapse Actually Looks Like:**

**Not this:**
- Image A: Δx = 0.0, μ = 0.05, xₚ = 0.48

**But this:**
- Image A: Δx = 0.0, μ = 0.05, xₚ = 0.48
- Image B: Δx = 0.0, μ = 0.05, xₚ = 0.48
- Image C: Δx = 0.0, μ = 0.05, xₚ = 0.48
- ... (600 more images with nearly identical coordinates)

A dataset where every image clusters within:
- Δx ∈ [-0.05, 0.05] (tight center clustering)
- μ ∈ [0.01, 0.08] (consistent fragmentation)
- θ ∈ [0.005, 0.015] (omnidirectional texture)

...exhibits **compositional monoculture**, even though the individual scores aren't "bad" in isolation. The diagnostic signal is **low variance**, not specific coordinate values.

---

## **VTL Is A Coordinate System, Not A Grading Rubric**

### **Think of it like measuring temperature:**

A thermometer reading "32°F" is not:
- ❌ "Good" or "bad"
- ❌ A quality judgment
- ❌ Pass/fail

It's simply: **"This system is at the freezing point of water."**

Whether that's desirable depends on context:
- Making ice cream: ✅ useful
- Growing tomatoes: ❌ problematic
- **Measuring compositional intent: ❓ neither - it's just data**

**VTL kernel values work the same way:** They measure **where** a composition sits in structural space, not whether it **should** sit there.

---

## **The Kernel Measures Authorship, Not Aesthetics**

### **Three valid compositional strategies:**

**Photographer 1 (Portraiture):**
- Δx = 0.0, μ = 0.85, xₚ = 0.35, θ = 0.65
- *(centered subjects, high cohesion, aligned)*

**Photographer 2 (Landscape):**
- Δx = -0.25, μ = 0.30, xₚ = 0.60, θ = 0.12
- *(rule-of-thirds, distributed elements, edge-engaged)*

**Photographer 3 (Street):**
- Δx = 0.15, μ = 0.15, xₚ = 0.55, θ = 0.08
- *(asymmetric, fragmented, peripheral activity)*

**All three are valid compositional choices.** VTL doesn't rank them - it **locates them in compositional space**.

---

## **Intentional Restraint vs. Unintentional Constraint**

**"Low" scores can represent discipline:**

**μ = 0.05 (low cohesion) could mean:**
- ✅ **Intentional:** Photographer chose fragmented composition to convey chaos (street photography, abstract texture)
- ❌ **Unintentional:** AI model failed to unify elements due to lack of compositional control

**θ = 0.01 (omnidirectional) could mean:**
- ✅ **Intentional:** Artist chose isotropic texture field (Jackson Pollock, natural patterns)
- ❌ **Unintentional:** AI model generated texture noise as default behavior

**VTL cannot distinguish intent from a single image.** Across a dataset, **variance patterns reveal authorship:**

- **High variance:** Different images explore different regions of compositional space = authorship
- **Low variance:** All images cluster tightly in one region = default/prior/collapse

---

## **The Diagnostic Power Is In The Distribution**

### **What VTL reveals about generative models:**

**Model A (Narrow Range):**
- Δx: mean = 0.02, std = 0.08 *(tight center clustering)*
- μ: mean = 0.04, std = 0.02 *(consistent fragmentation)*
- xₚ: mean = 0.47, std = 0.05 *(locked center/edge balance)*

**Interpretation:** Model explores only a small region of structural space.

**Model B (Broad Range):**
- Δx: mean = 0.05, std = 0.35 *(wide horizontal exploration)*
- μ: mean = 0.25, std = 0.30 *(spans fragmented → unified)*
- xₚ: mean = 0.52, std = 0.18 *(varied edge engagement)*

**Interpretation:** Model explores diverse structural strategies.

**Neither is "correct."** But if Model A's **semantic diversity** (CLIP embeddings) is high while its **compositional diversity** (VTL variance) is low, that reveals **decoupling between content and structure** - the hallmark of learned spatial priors.

---

## **What This Coordinate System Enables**

### **Three Novel Research Applications:**

**1. Cross-Model Comparison Without Aesthetic Bias**
- Compare Sora vs. MidJourney vs. Stable Diffusion on compositional range, not "quality"
- Measure whether models occupy different regions of structural space
- Detect systematic differences in spatial priors across architectures

**2. Detection of Learned Priors Via Cluster Analysis**
- Identify tight clustering in kernel space despite semantic diversity
- Quantify compositional monoculture as low variance in (Δx, μ, xₚ, θ, dₛ)
- Distinguish "model can't generate X" from "model defaults to Y regardless of prompt"

**3. Measurement of Prompt→Structure Fidelity**
- Test whether compositional prompts ("left-aligned", "fragmented", "radial") map to expected kernel coordinates
- Quantify the strength of spatial priors by measuring prompt-override behavior
- Detect when semantic controls fail to override structural defaults

This framing follows psychophysical measurement tradition (Stevens' power law, signal detection theory) where **objective localization precedes subjective evaluation**, enabling scientific comparison across systems without requiring agreement on aesthetic value.

---

## **Silence Is Also A Signal**

**If a model never produces:**
- Δx < -0.3 (strong left placement)
- μ > 0.7 (high cohesion single-subject)
- θ > 0.5 (strong directional alignment)

**This reveals:** The model's compositional vocabulary is **incomplete** - certain regions of structural space are unreachable, regardless of prompt.

Human photographers collectively explore the full range. If AI models cluster in one region, that's not because one region is "correct" - it's because the model has **learned a spatial prior** that constrains its compositional range.

---

## **VTL Measures Choice, Whatever The Choice May Be**

### **The fundamental premise: Composition is choice.**

- Where to place the center of mass (Δx)
- How to distribute structural unity (μ)
- Whether to engage the frame edges (xₚ)
- What orientation field to establish (θ)

**These choices are not scored as correct/incorrect. They're simply measured.**

The diagnostic power comes from asking:
- **Variance question:** Does this system explore diverse compositional coordinates?
- **Prompt-response question:** Do different semantic prompts map to different structural coordinates?
- **Cross-model question:** Do different generative systems occupy different regions of compositional space?

---

## **Final Framing**

**VTL Kernel Metrics is not:**
- A quality grader
- A compositional correctness checker
- A tool that says "center is bad"

**VTL Kernel Metrics is:**
- A coordinate system for compositional structure
- A tool for measuring diversity (or lack thereof) in spatial arrangements
- A detector of **decoupling** between semantic intent and structural realization

**The center isn't bad. Collapse is bad.**

**Authorship is variance. Priors are clustering.**

**VTL measures the shape of compositional space, not the value of individual points within it.**

---

### **Further Reading**

- Color space measurement: CIE L\*a\*b\* as coordinate system without aesthetic ranking
- Psychophysics: Stevens' Power Law (magnitude estimation independent of preference)
- Signal detection theory: Measuring system behavior independent of observer bias
- Compositional analysis: Arnheim's *Art and Visual Perception* (structural description vs. aesthetic judgment)

## **Note for # APPENDIX Absolute threshold for r_v (density measurement) ABOVE**

Canonical threshold: τ_abs = 0.15 (fixed for all measurements)

Sensitivity analysis: Threshold tested at {0.10, 0.15, 0.20} on Phase-1 Neutral
baseline (n=36). Results:

- τ = 0.10: r_v ∈ [0.82, 0.94], mean = 0.88, std = 0.032
- τ = 0.15: r_v ∈ [0.78, 0.85], mean = 0.81, std = 0.019  ← CANONICAL
- τ = 0.20: r_v ∈ [0.71, 0.79], mean = 0.75, std = 0.021

Selection rationale: 0.15 provides:
  • Mid-range coverage (avoids floor/ceiling effects)
  • Stable discrimination (sufficient variance)
  • Captures object edges while filtering texture noise

Compositional ranking stable across all three thresholds (Spearman ρ > 0.95).

In results:
- Visual field occupancy (texture + objects)
- mass_fraction (from the percentile mask) = “how much of the frame do we treat as ‘material’ under the top-band mask?” - It’s a bookkeeping artifact of the percentile-band choice, Because your percentile mask is defined by rank (e.g., ≥85th percentile), mass_fraction will be near-constant across images by construction.
- gradient_floor_85 (the 85th-percentile gradient magnitude) = “how strong is the gradient at the cutoff where the ‘top band’ begins?” Overall sharpness / texture / contrast baseline for the image’s gradient field.
  - High gradient_floor_85 ⇒ even the 85th percentile edge is already strong → lots of the image has high-frequency structure (texture, hard shadows, sharp lighting transitions, grain, etc.). Low gradient_floor_85 ⇒ most of the image is smooth/quiet; only a small tail contains strong edges. This number is telling you how “activated” the image is before you even talk about rᵥ.
- gradient_ceiling_97 (the 97th-percentile gradient magnitude) = “how strong are the very strongest edges?”
  - It tracks the peak edge strength / sharpening / hardest boundaries. So the pair (floor_85, ceiling_97) tells you whether the image is: uniformly edgy (floor high, ceiling moderately high), or mostly quiet but with a few extreme edges (floor low, ceiling still high).

A useful derived intuition is the tail gap: (ceiling_97 − floor_85)
Big gap = “quiet frame + a few punches.”Small gap = “edge energy everywhere.”

- Within-engine: rᵥ is valid and meaningful (especially as a delta vs that engine’s Neutral).
- Cross-engine: rᵥ is interpretable only with context (or with normalization).

Recommendation is to report rᵥ alongside (gradient_floor_85, gradient_ceiling_97) So every time rᵥ looks “wrong,” you can immediately see if it’s because the whole gradient field shifted.

One derived diagnostic scalar:

Edge Field Activation (EFA) = gradient_floor_85 or Tail Gap = gradient_ceiling_97 - gradient_floor_85

This gives a clean sentence like:

“This rᵥ drop is driven by global edge activation (high floor_85), not increased object packing.”